# DRO-PAS-MIL-CBM: Baselines and Ablation Study

This notebook evaluates DRO-PAS-MIL-CBM against CNN1D, ResNet1D, BiLSTM, TCN, and Transformer baselines using person-grouped cross-validation. It reports bootstrap comparisons, calibration, concept relevance, and component ablations.

The ablation sweep uses reduced settings by default. Adjust `ABLATION_N_FOLDS` and `ABLATION_MAX_EPOCHS` for the final experiment. Use `RUN_BASELINES`, `RUN_ABLATIONS`, and `BASELINE_MODEL_NAMES` to control runtime.

The loader expects the PhysioNet `gait-in-parkinsons-disease-1.0.0` directory.


In [1]:
# DRO-PAS-MIL-CBM experiment configuration

import os, re, glob, json, math, time, random, warnings
from pathlib import Path
from dataclasses import dataclass, asdict, replace
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GroupKFold
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAVE_SGKF = True
except Exception:
    HAVE_SGKF = False

from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix
)
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.pipeline import Pipeline

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")


# -------------------------
# Reproducibility
# -------------------------
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


SEED = 42
seed_everything(SEED)


# -------------------------
# Dataset / experiment config
# -------------------------
DATA_DIR = "../gait-in-parkinsons-disease-1.0.0/"
WINDOW_SIZE = 128
OVERLAP = 0.00
N_SENSORS = 16
N_SPLITS_TARGET = 5
VAL_PERSON_RATIO = 0.20

# Build a pool of windows per session, then the bag dataset samples from that pool.
MAX_POOL_WINDOWS_PER_SESSION = 96
TRAIN_BAG_WINDOWS = 32
EVAL_BAG_WINDOWS = 64

# DRO-PAS-MIL-CBM configuration
N_ANCHOR_CONCEPTS = 8
N_LEARNED_CONCEPTS = 8
K_CONCEPTS = N_ANCHOR_CONCEPTS + N_LEARNED_CONCEPTS
HIDDEN = 56
# Larger batches reduce noise in per-group loss estimates.
BAG_BATCH_SIZE = 24
MAX_EPOCHS = 70
PATIENCE = 12
LR = 8e-4
WEIGHT_DECAY = 1e-4
PRINT_EVERY_EPOCH = True

# Distributionally robust + multi-prototype enhancements.
N_PROTOTYPES_PER_CLASS = 3
DRO_CONF_BINS = 3
DRO_GROUPS = 2 * DRO_CONF_BINS
# EMA momentum for persistent Group-DRO weights.
DRO_EMA_MOMENTUM = 0.90

# Use one seed for debugging and three for the reported experiment.
ENSEMBLE_SEEDS = [42, 2026, 7]

# Threshold rule: avoid the very low sensitivity behavior observed in FSS-PGT-CBM.
MIN_VALIDATION_SENSITIVITY = 0.65

# Freeze hyperparameters before evaluating the outer test folds.

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, min(8, (os.cpu_count() or 2))))

# Deep-learning baselines
RUN_BASELINES = True
BASELINE_MODEL_NAMES = ["CNN1D", "ResNet1D", "BiLSTM", "TCN", "Transformer"]
BASELINE_SEED = 42
MAX_EPOCHS_BASELINE = 40
PATIENCE_BASELINE = 8

# Reduced ablation settings
RUN_ABLATIONS = True
ABLATION_N_FOLDS = 3
ABLATION_MAX_EPOCHS = 25
ABLATION_PATIENCE = 6

print("Device:", DEVICE)
print("sklearn:", sklearn.__version__)
print("StratifiedGroupKFold available:", HAVE_SGKF)
print("Torch threads:", torch.get_num_threads())

Device: cpu
sklearn: 1.5.1
StratifiedGroupKFold available: True
Torch threads: 8


In [2]:
# ============================================================
# 1. Data loading from gait-in-parkinsons-disease-1.0.0
#    Compatible with the supplied data loading.txt
# ============================================================

def resolve_data_dir(path_like: str) -> Path:
    candidates = [
        Path(path_like),
        Path("./gait-in-parkinsons-disease-1.0.0/"),
        Path("../gait-in-parkinsons-disease-1.0.0/"),
        Path("/mnt/data/gait-in-parkinsons-disease-1.0.0/"),
    ]
    for p in candidates:
        p = p.expanduser()
        if p.exists() and p.is_dir():
            return p
    raise FileNotFoundError(
        "Could not find gait-in-parkinsons-disease-1.0.0. "
        "Place the folder beside this notebook or update DATA_DIR."
    )


DATA_DIR = resolve_data_dir(DATA_DIR)
OUT_DIR = DATA_DIR / "experiment_outputs_dro_pas_mil_cbm"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR.resolve())
print("OUT_DIR :", OUT_DIR.resolve())


def infer_label(filename: str) -> Optional[int]:
    name = os.path.basename(filename).lower()
    if "co" in name:
        return 0
    if "pt" in name:
        return 1
    return None


def infer_subject_id(filename: str) -> str:
    stem = Path(filename).stem.lower()
    # PhysioNet gait files commonly encode group + subject + trial in filename.
    # This pattern intentionally groups repeated trials of the same subject.
    m = re.match(r"^([a-z]+(?:co|pt)\d+)", stem)
    if m:
        return m.group(1)
    m = re.match(r"^(.+?)_\d+$", stem)
    if m:
        return m.group(1)
    return stem


def load_single_file(fp: str) -> Tuple[np.ndarray, int]:
    df = pd.read_csv(fp, sep="\t", header=None)
    X = df.iloc[:, 1:17].values
    if X.shape[1] != N_SENSORS:
        raise ValueError(f"Expected {N_SENSORS} sensor columns, got {X.shape[1]}")
    if np.isnan(X).any():
        X = np.nan_to_num(X, nan=0.0)
    label = infer_label(fp)
    if label is None:
        raise ValueError(f"Cannot infer label from filename: {fp}")
    return X.astype(np.float32), int(label)


def load_dataset(folder: Path) -> List[Dict]:
    files = sorted(glob.glob(str(folder / "*.txt")))
    records = []
    for fp in files:
        try:
            X, label = load_single_file(fp)
            records.append({
                "file_path": fp,
                "session_id": Path(fp).stem,
                "person_id": infer_subject_id(fp),
                "label": label,
                "X": X,
                "n_frames": int(X.shape[0]),
            })
        except Exception as e:
            print(f"Skipped {Path(fp).name}: {e}")
    if not records:
        raise RuntimeError("No usable gait TXT files were loaded.")
    labels = np.array([r["label"] for r in records])
    print(f"Loaded {len(records)} files | class dist [Control, PD]: {np.bincount(labels, minlength=2)}")
    return records


records = load_dataset(DATA_DIR)
sessions_df = pd.DataFrame([{k: v for k, v in r.items() if k != "X"} for r in records])

print("\nDataset sanity report")
print("Recordings:", len(sessions_df))
print("Persons   :", sessions_df["person_id"].nunique())
print("Sessions per class:")
print(sessions_df["label"].value_counts().rename(index={0: "Control", 1: "PD"}))
print("\nFrames per recording:")
display(sessions_df["n_frames"].describe().to_frame().T)

person_labels = sessions_df.groupby("person_id")["label"].max()
print("\nPersons per class:")
print(person_labels.value_counts().rename(index={0: "Control", 1: "PD"}))

DATA_DIR: D:\2 ZUJ.EDU.JO\My research\REMAP- Parkinson-Open dataset\Paper 2\Rivision1\gait-in-parkinsons-disease-1.0.0
OUT_DIR : D:\2 ZUJ.EDU.JO\My research\REMAP- Parkinson-Open dataset\Paper 2\Rivision1\gait-in-parkinsons-disease-1.0.0\experiment_outputs_dro_pas_mil_cbm
Loaded 306 files | class dist [Control, PD]: [ 92 214]

Dataset sanity report
Recordings: 306
Persons   : 165
Sessions per class:
label
PD         214
Control     92
Name: count, dtype: int64

Frames per recording:


,count,mean,std,min,25%,50%,75%,max
n_frames,306.0,10841.666667,2582.981934,4034.0,9187.0,12119.0,12119.0,26366.0



Persons per class:
label
PD         93
Control    72
Name: count, dtype: int64


In [3]:
# ============================================================
# 2. Leakage-safe grouped splits
# ============================================================

def make_grouped_splits(sessions_df: pd.DataFrame, n_splits_target: int = 5, seed: int = 42):
    y = sessions_df["label"].to_numpy(dtype=int)
    groups = sessions_df["person_id"].to_numpy()
    person_label = sessions_df.groupby("person_id")["label"].max()
    min_class_persons = int(person_label.value_counts().min())
    n_splits = min(n_splits_target, min_class_persons)
    if n_splits < 2:
        raise RuntimeError("Need at least two subjects in each class for grouped CV.")
    if HAVE_SGKF:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        splits = list(splitter.split(np.zeros(len(y)), y, groups))
    else:
        splitter = GroupKFold(n_splits=n_splits)
        splits = list(splitter.split(np.zeros(len(y)), y, groups))
    return splits


SPLITS = make_grouped_splits(sessions_df, N_SPLITS_TARGET, SEED)
print(f"Using {len(SPLITS)} grouped folds.")
for i, (_, te) in enumerate(SPLITS, 1):
    print(
        f"Fold {i}: test sessions={len(te)}, "
        f"test persons={sessions_df.iloc[te]['person_id'].nunique()}, "
        f"PD prevalence={sessions_df.iloc[te]['label'].mean():.3f}"
    )


def train_val_split_by_person(train_records: List[Dict], val_ratio: float = 0.20, seed: int = 42):
    person_to_label = {}
    for r in train_records:
        person_to_label[r["person_id"]] = max(person_to_label.get(r["person_id"], 0), int(r["label"]))
    persons = np.array(sorted(person_to_label.keys()))
    labels = np.array([person_to_label[p] for p in persons], dtype=int)
    try:
        p_tr, p_va = train_test_split(persons, test_size=val_ratio, random_state=seed, stratify=labels)
    except Exception:
        rng = np.random.RandomState(seed)
        persons_shuf = persons.copy()
        rng.shuffle(persons_shuf)
        n_va = max(1, int(round(len(persons_shuf) * val_ratio)))
        p_va, p_tr = persons_shuf[:n_va], persons_shuf[n_va:]
    p_tr, p_va = set(p_tr), set(p_va)
    tr = [r for r in train_records if r["person_id"] in p_tr]
    va = [r for r in train_records if r["person_id"] in p_va]
    return tr, va


def fit_scaler(train_records: List[Dict]) -> StandardScaler:
    scaler = StandardScaler()
    for r in train_records:
        scaler.partial_fit(r["X"])
    return scaler


def get_records_by_indices(records: List[Dict], indices: np.ndarray) -> List[Dict]:
    return [records[int(i)] for i in indices]

Using 5 grouped folds.
Fold 1: test sessions=65, test persons=32, PD prevalence=0.708
Fold 2: test sessions=47, test persons=32, PD prevalence=0.660
Fold 3: test sessions=61, test persons=33, PD prevalence=0.754
Fold 4: test sessions=73, test persons=34, PD prevalence=0.712
Fold 5: test sessions=60, test persons=34, PD prevalence=0.650


In [4]:
# ============================================================
# 3. Metrics, thresholds, and static features
# ============================================================

def safe_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_score))


def safe_ap(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(average_precision_score(y_true, y_score))


def confusion_parts(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / max(1, tp + fn)
    spec = tn / max(1, tn + fp)
    return tn, fp, fn, tp, sens, spec


def threshold_youden_with_sensitivity_floor(y_true, p, min_sens: float = 0.65):
    """Choose a threshold on validation data.

    Primary rule: maximize balanced accuracy among thresholds satisfying the
    requested sensitivity floor. Fallback: Youden threshold when the floor is
    impossible on that fold.
    """
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    if len(np.unique(y_true)) < 2:
        return 0.5
    cand = np.unique(np.quantile(p, np.linspace(0.02, 0.98, 97)))
    best_thr, best_score = 0.5, -1e9
    feasible = False
    fallback_thr, fallback_j = 0.5, -1e9
    for thr in cand:
        yp = (p >= thr).astype(int)
        _, _, _, _, sens, spec = confusion_parts(y_true, yp)
        bal = 0.5 * (sens + spec)
        j = sens + spec - 1.0
        if j > fallback_j:
            fallback_j, fallback_thr = j, float(thr)
        if sens >= min_sens:
            feasible = True
            # mild penalty for too-low specificity, but sensitivity floor is enforced first
            score = bal - 0.05 * max(0.0, 0.45 - spec)
            if score > best_score:
                best_score, best_thr = score, float(thr)
    return best_thr if feasible else fallback_thr


def compute_metrics_with_threshold(y_true, p, thr=0.5, prefix=""):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    y_pred = (p >= thr).astype(int)
    return compute_metrics_with_predictions(y_true, p, y_pred, prefix=prefix)


def compute_metrics_with_predictions(y_true, p, y_pred, prefix=""):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    y_pred = np.asarray(y_pred).astype(int)
    tn, fp, fn, tp, sens, spec = confusion_parts(y_true, y_pred)
    d = {
        "AUC": safe_auc(y_true, p),
        "AP": safe_ap(y_true, p),
        "Acc": float(accuracy_score(y_true, y_pred)),
        "Prec": float(precision_score(y_true, y_pred, zero_division=0)),
        "Rec": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Spec": float(spec),
        "Sens": float(sens),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }
    return {prefix + k: v for k, v in d.items()}


def extract_record_features(X, use_velocity=True):
    X = np.asarray(X, dtype=np.float32)
    left, right = X[:, :8], X[:, 8:16]
    total = left.sum(axis=1) + right.sum(axis=1)
    lr_asym = (left.sum(axis=1) - right.sum(axis=1)) / (np.abs(total) + 1e-6)
    feats = [
        np.nanmean(X, axis=0), np.nanstd(X, axis=0),
        np.nanmin(X, axis=0), np.nanmax(X, axis=0), np.nanmedian(X, axis=0),
        np.nanpercentile(X, 25, axis=0), np.nanpercentile(X, 75, axis=0),
        np.array([np.mean(total), np.std(total), np.mean(np.abs(lr_asym)), np.std(lr_asym)], dtype=np.float32),
    ]
    if use_velocity and len(X) > 1:
        V = np.diff(X, axis=0)
        feats += [
            np.nanmean(V, axis=0), np.nanstd(V, axis=0),
            np.nanmin(V, axis=0), np.nanmax(V, axis=0), np.nanmedian(V, axis=0),
        ]
    return np.nan_to_num(np.concatenate(feats), nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


X_static = np.vstack([extract_record_features(r["X"]) for r in records])
y_static = sessions_df["label"].to_numpy(dtype=int)
groups_static = sessions_df["person_id"].to_numpy()
session_ids_static = sessions_df["session_id"].to_numpy()


# NOTE: the standalone RandomForest "static baseline" model has been removed
# per request (superseded by direct comparison against five SOTA deep
# learning baselines in cells 6-8). X_static / y_static / groups_static /
# session_ids_static are still built here because extract_record_features()
# is also used internally for the static-feature fusion branch and the
# ExtraTrees teacher used for DRO-group assignment / distillation -- those
# are part of the proposed model's own architecture, not a competing
# baseline, so they stay.


def paired_bootstrap_auc_test(y_true, p_a, p_b, groups, n_boot: int = 2000, seed: int = 42):
    """Person-level paired bootstrap comparing two sets of session scores.

    Resamples PERSONS (not sessions) with replacement so the resampling
    respects the same grouping used for the CV splits, then compares AUC(a)
    vs AUC(b) on each resample. Returns the observed AUC difference, a 95%
    percentile CI for the difference, and a two-sided bootstrap p-value for
    "AUC_a == AUC_b". This is what should be used to back up any claim that
    the proposed model beats (or loses to) the baseline, rather than quoting
    single-point AUC numbers.
    """
    y_true = np.asarray(y_true).astype(int)
    p_a = np.asarray(p_a).astype(float)
    p_b = np.asarray(p_b).astype(float)
    groups = np.asarray(groups)
    persons = np.unique(groups)
    rng = np.random.RandomState(seed)

    diffs = []
    for _ in range(n_boot):
        sample_persons = rng.choice(persons, size=len(persons), replace=True)
        idx = np.concatenate([np.where(groups == pid)[0] for pid in sample_persons])
        yt = y_true[idx]
        if len(np.unique(yt)) < 2:
            continue
        auc_a = roc_auc_score(yt, p_a[idx])
        auc_b = roc_auc_score(yt, p_b[idx])
        diffs.append(auc_a - auc_b)
    diffs = np.asarray(diffs)
    obs_diff = safe_auc(y_true, p_a) - safe_auc(y_true, p_b)
    lo, hi = np.percentile(diffs, [2.5, 97.5]) if len(diffs) else (np.nan, np.nan)
    # two-sided bootstrap p-value: fraction of resamples on the other side of 0
    if len(diffs):
        p_value = 2.0 * min(np.mean(diffs <= 0.0), np.mean(diffs >= 0.0))
        p_value = float(min(1.0, p_value))
    else:
        p_value = np.nan
    return {
        "auc_diff": float(obs_diff),
        "ci95_lo": float(lo),
        "ci95_hi": float(hi),
        "p_value": p_value,
        "n_boot_valid": int(len(diffs)),
    }


def fit_static_scaler(train_records: List[Dict]) -> StandardScaler:
    """Fit on TRAIN-fold static features only, mirroring fit_scaler for the
    raw sensor scaler. Used to standardize the static-feature fusion branch
    fed into the deep model (see build_bags / DROMultiPrototypePASMILCBM)."""
    Xtr = np.vstack([extract_record_features(r["X"]) for r in train_records])
    scaler = StandardScaler()
    scaler.fit(Xtr)
    return scaler


def fit_static_teacher(train_records: List[Dict]):
    Xtr = np.vstack([extract_record_features(r["X"]) for r in train_records])
    ytr = np.array([r["label"] for r in train_records], dtype=int)
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("et", ExtraTreesClassifier(n_estimators=500, min_samples_leaf=2,
                                    class_weight="balanced", random_state=SEED, n_jobs=-1)),
    ])
    clf.fit(Xtr, ytr)
    return clf

In [5]:
# ============================================================
# 4. Build session bags
# ============================================================

def make_window_pool_for_record(r: Dict, scaler: StandardScaler, max_windows: int = 96,
                                window: int = 128, overlap: float = 0.0):
    Xs = scaler.transform(r["X"]).astype(np.float32)
    n = len(Xs)
    if n < window:
        return None
    step = max(1, int(window * (1.0 - overlap)))
    starts = np.arange(0, n - window + 1, step, dtype=int)
    if len(starts) > max_windows:
        idx = np.linspace(0, len(starts) - 1, max_windows).round().astype(int)
        starts = starts[np.unique(idx)]
    windows = np.stack([Xs[s:s + window] for s in starts]).astype(np.float32)
    return windows


def assign_dro_group(label: int, teacher_prob: float) -> int:
    """Create robust-training groups from class x teacher-confidence tier.

    The aim is not to encode private identities, but to avoid optimizing only
    the average session. The groups capture easy-control/easy-PD/ambiguous cases
    so the loss can emphasize the worst subgroup in each mini-batch.
    """
    if teacher_prob < 0.35:
        tier = 0
    elif teacher_prob > 0.65:
        tier = 2
    else:
        tier = 1
    return int(label) * DRO_CONF_BINS + tier


def build_bags(records_subset: List[Dict], scaler: StandardScaler,
               teacher_model=None, max_pool_windows: int = 96,
               static_scaler: Optional[StandardScaler] = None):
    """Build MIL bags; fit ``static_scaler`` on training records only."""
    bags = []
    n_win = []
    for r in records_subset:
        pool = make_window_pool_for_record(r, scaler, max_pool_windows, WINDOW_SIZE, OVERLAP)
        if pool is None:
            continue
        static_feat = extract_record_features(r["X"])
        if static_scaler is not None:
            static_feat = static_scaler.transform(static_feat.reshape(1, -1)).ravel().astype(np.float32)
        teacher_prob = 0.5
        if teacher_model is not None:
            f = extract_record_features(r["X"]).reshape(1, -1)
            teacher_prob = float(teacher_model.predict_proba(f)[0, 1])
        dro_group = assign_dro_group(int(r["label"]), teacher_prob)
        bags.append({
            "X_pool": pool,
            "static_feat": static_feat.astype(np.float32),
            "label": int(r["label"]),
            "session_id": r["session_id"],
            "person_id": r["person_id"],
            "teacher_prob": float(teacher_prob),
            "dro_group": int(dro_group),
            "n_pool_windows": int(pool.shape[0]),
        })
        n_win.append(pool.shape[0])
    if not bags:
        raise RuntimeError("No bags were built. Check WINDOW_SIZE and data length.")
    print(
        f"  Bags: sessions={len(bags)}, mean_pool_windows={np.mean(n_win):.1f}, "
        f"min={np.min(n_win)}, max={np.max(n_win)}"
    )
    return bags


class GaitBagDataset(Dataset):
    def __init__(self, bags: List[Dict], bag_windows: int = 32, train: bool = False,
                 noise_std: float = 0.020, sensor_drop_prob: float = 0.08,
                 time_mask_prob: float = 0.15, amp_range=(0.90, 1.10)):
        self.bags = bags
        self.bag_windows = bag_windows
        self.train = train
        self.noise_std = noise_std
        self.sensor_drop_prob = sensor_drop_prob
        self.time_mask_prob = time_mask_prob
        self.amp_range = amp_range

    def __len__(self):
        return len(self.bags)

    def _select_windows(self, pool):
        n = pool.shape[0]
        if self.train:
            idx = np.random.choice(n, size=self.bag_windows, replace=(n < self.bag_windows))
        else:
            if n >= self.bag_windows:
                idx = np.linspace(0, n - 1, self.bag_windows).round().astype(int)
            else:
                extra = np.random.choice(n, size=self.bag_windows - n, replace=True)
                idx = np.concatenate([np.arange(n), extra])
        x = pool[idx].copy()
        return x

    def _augment(self, x):
        if not self.train:
            return x
        x += np.random.normal(0, self.noise_std, x.shape).astype(np.float32)
        x *= np.float32(np.random.uniform(*self.amp_range))
        # Drop one whole sensor in some bags.
        if np.random.rand() < self.sensor_drop_prob:
            c = np.random.randint(0, x.shape[-1])
            x[:, :, c] = 0.0
        # Mask a short time span in some windows.
        if np.random.rand() < self.time_mask_prob:
            T = x.shape[1]
            w = np.random.randint(max(2, T // 16), max(3, T // 6))
            s = np.random.randint(0, max(1, T - w))
            x[:, s:s + w, :] = 0.0
        return x

    def __getitem__(self, idx):
        b = self.bags[idx]
        x = self._augment(self._select_windows(b["X_pool"]))
        static = b["static_feat"].copy()
        if self.train:
            # Light noise on the static branch too, so the model can't just
            # memorize the exact RF-style feature vector per training session.
            static = static + np.random.normal(0, 0.03, static.shape).astype(np.float32)
        return {
            "x": torch.tensor(x, dtype=torch.float32),
            "static": torch.tensor(static, dtype=torch.float32),
            "y": torch.tensor(float(b["label"]), dtype=torch.float32),
            "teacher": torch.tensor(float(b["teacher_prob"]), dtype=torch.float32),
            "group": torch.tensor(int(b.get("dro_group", 0)), dtype=torch.long),
            "session_id": b["session_id"],
            "person_id": b["person_id"],
        }

In [6]:
# ============================================================
# 5. Sparsemax and DRO-PAS-MIL-CBM model
# ============================================================

def sparsemax(logits, dim=-1):
    logits = logits - logits.max(dim=dim, keepdim=True).values
    zs = torch.sort(logits, descending=True, dim=dim).values
    rhos = torch.arange(1, logits.size(dim) + 1, device=logits.device, dtype=logits.dtype)
    view = [1] * logits.dim()
    view[dim] = -1
    rhos = rhos.view(view)
    cumsum_zs = zs.cumsum(dim)
    support = 1 + rhos * zs > cumsum_zs
    k = support.sum(dim=dim, keepdim=True).clamp(min=1)
    tau = (cumsum_zs.gather(dim, k - 1) - 1) / k.to(logits.dtype)
    return torch.clamp(logits - tau, min=0.0)


class DepthwiseSeparableConv1d(nn.Module):
    def __init__(self, channels, kernel_size=7, dilation=1, dropout=0.10):
        super().__init__()
        padding = dilation * (kernel_size // 2)
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=padding,
                      dilation=dilation, groups=channels, bias=False),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Conv1d(channels, channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.block(x)


class LightweightTemporalBlock(nn.Module):
    def __init__(self, channels, dropout=0.10):
        super().__init__()
        self.b1 = DepthwiseSeparableConv1d(channels, 3, 1, dropout)
        self.b2 = DepthwiseSeparableConv1d(channels, 7, 2, dropout)
        self.fuse = nn.Sequential(
            nn.Conv1d(channels * 2, channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.fuse(torch.cat([self.b1(x), self.b2(x)], dim=1))


class WindowTemporalEncoder(nn.Module):
    def __init__(self, n_sensors=16, hidden=56, dropout=0.15):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(n_sensors, hidden, kernel_size=1, bias=False),
            nn.BatchNorm1d(hidden),
            nn.GELU(),
        )
        self.temporal = nn.Sequential(
            LightweightTemporalBlock(hidden, dropout=dropout),
            LightweightTemporalBlock(hidden, dropout=dropout),
            LightweightTemporalBlock(hidden, dropout=dropout),
        )
        self.pool_score = nn.Sequential(
            nn.Conv1d(hidden, max(4, hidden // 2), 1),
            nn.Tanh(),
            nn.Conv1d(max(4, hidden // 2), 1, 1),
        )

    def forward(self, x):
        # x: [B, N, T, C]
        B, N, T, C = x.shape
        z = x.reshape(B * N, T, C).transpose(1, 2)  # [BN, C, T]
        h = self.temporal(self.stem(z))
        a = torch.softmax(self.pool_score(h), dim=-1)
        mean = (a * h).sum(dim=-1)
        var = (a * (h - mean.unsqueeze(-1)) ** 2).sum(dim=-1)
        std = torch.sqrt(var + 1e-6)
        feat = torch.cat([mean, std], dim=1)
        return feat.reshape(B, N, -1)


class PhysiologicalAnchorLayer(nn.Module):
    """Differentiable window-level anchor concepts from the 16 force sensors.

    Sensors are split as first 8 vs last 8, corresponding to the two feet in
    the common VGRF organization. The anchors are normalized by LayerNorm and
    squashed to [-1, 1] for concept-bottleneck use.
    """
    def __init__(self):
        super().__init__()
        self.norm = nn.LayerNorm(N_ANCHOR_CONCEPTS)

    def forward(self, x):
        # x: [B, N, T, 16]
        eps = 1e-5
        left = x[..., :8].sum(dim=-1)       # [B,N,T]
        right = x[..., 8:16].sum(dim=-1)    # [B,N,T]
        total = left + right
        lr = left - right
        denom = total.abs() + eps
        asym = lr / denom

        d_total = total[..., 1:] - total[..., :-1]
        d_left = left[..., 1:] - left[..., :-1]
        d_right = right[..., 1:] - right[..., :-1]

        anchors = torch.stack([
            total.mean(dim=-1),                                  # C01: total load level
            total.std(dim=-1, unbiased=False),                   # C02: total load variability
            total.std(dim=-1, unbiased=False) / (total.abs().mean(dim=-1) + eps),  # C03: coefficient of variation
            asym.abs().mean(dim=-1),                             # C04: absolute left-right asymmetry
            asym.std(dim=-1, unbiased=False),                    # C05: asymmetry variability
            asym.mean(dim=-1),                                   # C06: signed left-right dominance
            (d_total ** 2).mean(dim=-1),                         # C07: temporal force-change energy
            ((d_left - d_right).abs() / (d_total.abs() + eps)).mean(dim=-1),       # C08: bilateral change mismatch
        ], dim=-1)
        return torch.tanh(self.norm(anchors))


ANCHOR_NAMES = [
    "C01_total_load_level",
    "C02_total_load_variability",
    "C03_total_load_CV",
    "C04_abs_left_right_asymmetry",
    "C05_asymmetry_variability",
    "C06_signed_left_right_dominance",
    "C07_force_change_energy",
    "C08_bilateral_change_mismatch",
]
LEARNED_NAMES = [f"C{N_ANCHOR_CONCEPTS+i+1:02d}_learned_temporal_{i+1}" for i in range(N_LEARNED_CONCEPTS)]
CONCEPT_NAMES = ANCHOR_NAMES + LEARNED_NAMES


class DROMultiPrototypePASMILCBM(nn.Module):
    """DRO-PAS-MIL-CBM with multi-prototype and anchor-supervised decisions.

    Novelty-oriented design choices:
      1. Multi-instance session learning: one recording = one bag of gait windows.
      2. Explicit physiological anchors + learned temporal concepts.
      3. Sparse concept relevance and anchor-mass regularization.
      4. Multiple Control/PD prototypes in concept space for heterogeneous gait patterns.
      5. Separate anchor-only auxiliary logit to prevent learned-concept dominance.
    """
    def __init__(self, n_sensors=16, hidden=56, n_learned=8, dropout=0.15,
                 n_prototypes_per_class: int = N_PROTOTYPES_PER_CLASS,
                 static_feat_dim: int = 1, use_static: bool = True):
        super().__init__()
        self.use_static = use_static
        self.anchor = PhysiologicalAnchorLayer()
        self.encoder = WindowTemporalEncoder(n_sensors=n_sensors, hidden=hidden, dropout=dropout)
        self.learned_head = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_learned),
            nn.Tanh(),
        )
        self.window_attention = nn.Sequential(
            nn.Linear(N_ANCHOR_CONCEPTS + n_learned, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1),
        )
        self.relevance_head = nn.Linear(N_ANCHOR_CONCEPTS + n_learned, N_ANCHOR_CONCEPTS + n_learned)
        self.concept_effect = nn.Parameter(torch.randn(N_ANCHOR_CONCEPTS + n_learned) * 0.04)
        self.bias = nn.Parameter(torch.zeros(1))

        # Multiple prototypes per class allow more than one PD/control gait subtype.
        self.prototypes = nn.Parameter(torch.randn(2, n_prototypes_per_class, N_ANCHOR_CONCEPTS + n_learned) * 0.10)
        self.proto_scale = nn.Parameter(torch.tensor(1.0))
        self.proto_tau = nn.Parameter(torch.tensor(0.20))

        # Anchor-only auxiliary head: used both in final prediction and auxiliary loss.
        self.anchor_head = nn.Sequential(
            nn.LayerNorm(N_ANCHOR_CONCEPTS),
            nn.Linear(N_ANCHOR_CONCEPTS, max(4, N_ANCHOR_CONCEPTS // 2)),
            nn.GELU(),
            nn.Linear(max(4, N_ANCHOR_CONCEPTS // 2), 1),
        )

        # Static session-level evidence complements the sampled MIL windows.
        self.static_head = nn.Sequential(
            nn.LayerNorm(static_feat_dim),
            nn.Linear(static_feat_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Linear(hidden // 2, 1),
        )

        # Learn mixture of CBM, prototype, anchor-only, and static evidence.
        self.mix_logits = nn.Parameter(torch.zeros(4))

    def _softmin_proto_dist(self, bag_concepts):
        # bag_concepts: [B,K]; prototypes: [2,P,K]
        all_dists = ((bag_concepts[:, None, None, :] - self.prototypes[None, :, :, :]) ** 2).mean(dim=-1)  # [B,2,P]
        tau = self.proto_tau.abs().clamp(0.05, 1.50)
        softmin = -tau * torch.logsumexp(-all_dists / tau, dim=2)  # [B,2]
        min_dists = all_dists.min(dim=2).values
        return softmin, min_dists, all_dists

    def logit_from_concepts(self, bag_concepts, static_feat):
        relevance = sparsemax(self.relevance_head(bag_concepts), dim=1)
        cbm_logit = (relevance * bag_concepts * self.concept_effect.view(1, -1)).sum(dim=1, keepdim=True) + self.bias

        soft_dists, min_dists, all_proto_dists = self._softmin_proto_dist(bag_concepts)
        proto_logit = self.proto_scale.clamp(0.1, 20.0) * (soft_dists[:, 0:1] - soft_dists[:, 1:2])
        anchor_logit = self.anchor_head(bag_concepts[:, :N_ANCHOR_CONCEPTS])

        if self.use_static:
            static_logit = self.static_head(static_feat)
            mix = torch.softmax(self.mix_logits, dim=0)
            logit = mix[0] * cbm_logit + mix[1] * proto_logit + mix[2] * anchor_logit + mix[3] * static_logit
        else:
            # Ablation: static-fusion branch disabled. Mixture renormalized
            # over the remaining three (still-interpretable) experts only;
            # static_logit is still computed for logging shape-compatibility
            # but contributes nothing to the prediction.
            static_logit = self.static_head(static_feat).detach() * 0.0
            mix3 = torch.softmax(self.mix_logits[:3], dim=0)
            logit = mix3[0] * cbm_logit + mix3[1] * proto_logit + mix3[2] * anchor_logit
            mix = torch.cat([mix3, torch.zeros(1, device=mix3.device, dtype=mix3.dtype)])

        reliability = torch.sigmoid(torch.abs(proto_logit.detach()).squeeze(1))
        return logit, cbm_logit, proto_logit, anchor_logit, static_logit, min_dists, all_proto_dists, relevance, reliability, mix

    def forward(self, x, static_feat):
        # x: [B, N, T, C]; static_feat: [B, D]
        anchor_concepts = self.anchor(x)                     # [B,N,8]
        feat = self.encoder(x)                               # [B,N,2H]
        learned_concepts = self.learned_head(feat)            # [B,N,8]
        window_concepts = torch.cat([anchor_concepts, learned_concepts], dim=-1)
        attn_logits = self.window_attention(window_concepts).squeeze(-1)
        attn = torch.softmax(attn_logits, dim=1)
        bag_concepts = (attn.unsqueeze(-1) * window_concepts).sum(dim=1)
        (logit, cbm_logit, proto_logit, anchor_logit, static_logit,
         dists, all_dists, relevance, reliability, mix) = self.logit_from_concepts(bag_concepts, static_feat)
        return {
            "logit": logit,
            "cbm_logit": cbm_logit,
            "proto_logit": proto_logit,
            "anchor_logit": anchor_logit,
            "static_logit": static_logit,
            "prototype_dists": dists,
            "all_prototype_dists": all_dists,
            "bag_concepts": bag_concepts,
            "window_concepts": window_concepts,
            "relevance": relevance,
            "attention": attn,
            "reliability": reliability,
            "mix": mix,
        }

# ============================================================
# 5b. Deep-learning baselines
# ============================================================
# Each encoder returns window embeddings with shape [B, N, D].
# ============================================================

class CNN1DEncoder(nn.Module):
    def __init__(self, n_sensors=16, hidden=64, dropout=0.20):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_sensors, hidden, kernel_size=7, padding=3), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Conv1d(hidden, hidden, kernel_size=5, padding=2), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(hidden, hidden, kernel_size=3, padding=1), nn.BatchNorm1d(hidden), nn.ReLU(),
        )
        self.out_dim = hidden

    def forward(self, x):
        B, N, T, C = x.shape
        z = x.reshape(B * N, T, C).transpose(1, 2)
        h = self.net(z)
        emb = h.mean(dim=-1)
        return emb.reshape(B, N, -1)


class ResBlock1D(nn.Module):
    def __init__(self, ch, dropout=0.10):
        super().__init__()
        self.conv1 = nn.Conv1d(ch, ch, 7, padding=3)
        self.bn1 = nn.BatchNorm1d(ch)
        self.conv2 = nn.Conv1d(ch, ch, 5, padding=2)
        self.bn2 = nn.BatchNorm1d(ch)
        self.conv3 = nn.Conv1d(ch, ch, 3, padding=1)
        self.bn3 = nn.BatchNorm1d(ch)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = self.act(self.bn1(self.conv1(x)))
        h = self.act(self.bn2(self.conv2(h)))
        h = self.bn3(self.conv3(h))
        return self.drop(self.act(x + h))


class ResNet1DEncoder(nn.Module):
    def __init__(self, n_sensors=16, hidden=64, n_blocks=3, dropout=0.10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(n_sensors, hidden, 1), nn.BatchNorm1d(hidden), nn.ReLU())
        self.blocks = nn.Sequential(*[ResBlock1D(hidden, dropout) for _ in range(n_blocks)])
        self.out_dim = hidden

    def forward(self, x):
        B, N, T, C = x.shape
        z = x.reshape(B * N, T, C).transpose(1, 2)
        h = self.blocks(self.stem(z))
        emb = h.mean(dim=-1)
        return emb.reshape(B, N, -1)


class BiLSTMEncoder(nn.Module):
    def __init__(self, n_sensors=16, hidden=48, num_layers=2, dropout=0.20):
        super().__init__()
        self.lstm = nn.LSTM(n_sensors, hidden, num_layers=num_layers, batch_first=True,
                             bidirectional=True, dropout=dropout if num_layers > 1 else 0.0)
        self.out_dim = hidden * 2

    def forward(self, x):
        B, N, T, C = x.shape
        z = x.reshape(B * N, T, C)
        out, _ = self.lstm(z)
        emb = out.mean(dim=1)
        return emb.reshape(B, N, -1)


class TCNBlock(nn.Module):
    def __init__(self, ch, kernel_size=5, dilation=1, dropout=0.15):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(ch, ch, kernel_size, padding=self.pad, dilation=dilation)
        self.bn = nn.BatchNorm1d(ch)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = self.conv(x)
        if self.pad > 0:
            h = h[..., :-self.pad]  # causal: drop the future-leaking tail from padding
        h = self.drop(self.act(self.bn(h)))
        return x + h


class TCNEncoder(nn.Module):
    def __init__(self, n_sensors=16, hidden=64, n_blocks=4, dropout=0.15):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(n_sensors, hidden, 1), nn.BatchNorm1d(hidden), nn.ReLU())
        self.blocks = nn.Sequential(*[
            TCNBlock(hidden, kernel_size=5, dilation=2 ** i, dropout=dropout) for i in range(n_blocks)
        ])
        self.out_dim = hidden

    def forward(self, x):
        B, N, T, C = x.shape
        z = x.reshape(B * N, T, C).transpose(1, 2)
        h = self.blocks(self.stem(z))
        emb = h.mean(dim=-1)
        return emb.reshape(B, N, -1)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerBackboneEncoder(nn.Module):
    def __init__(self, n_sensors=16, hidden=48, n_heads=4, n_layers=2, dropout=0.15):
        super().__init__()
        self.input_proj = nn.Linear(n_sensors, hidden)
        self.pos = PositionalEncoding(hidden)
        layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads, dim_feedforward=hidden * 2,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.out_dim = hidden

    def forward(self, x):
        B, N, T, C = x.shape
        z = x.reshape(B * N, T, C)
        z = self.pos(self.input_proj(z))
        h = self.encoder(z)
        emb = h.mean(dim=1)
        return emb.reshape(B, N, -1)


class MILAttentionPool(nn.Module):
    """Standard attention-based MIL pooling (Ilse et al., 2018), shared by
    every baseline so only the encoder backbone varies across comparisons."""
    def __init__(self, dim, hidden=32):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, window_emb):
        scores = self.attn(window_emb).squeeze(-1)
        w = torch.softmax(scores, dim=1)
        bag_emb = (w.unsqueeze(-1) * window_emb).sum(dim=1)
        return bag_emb, w


class MILBaselineClassifier(nn.Module):
    def __init__(self, encoder, dropout=0.20):
        super().__init__()
        self.encoder = encoder
        self.pool = MILAttentionPool(encoder.out_dim)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(encoder.out_dim, 1))

    def forward(self, x, static_feat=None):
        window_emb = self.encoder(x)
        bag_emb, attn = self.pool(window_emb)
        logit = self.head(bag_emb)
        return {"logit": logit, "attention": attn}


def build_baseline_model(name: str, n_sensors: int = N_SENSORS) -> "MILBaselineClassifier":
    if name == "CNN1D":
        enc = CNN1DEncoder(n_sensors=n_sensors, hidden=64, dropout=0.20)
    elif name == "ResNet1D":
        enc = ResNet1DEncoder(n_sensors=n_sensors, hidden=64, n_blocks=3, dropout=0.10)
    elif name == "BiLSTM":
        enc = BiLSTMEncoder(n_sensors=n_sensors, hidden=48, num_layers=2, dropout=0.20)
    elif name == "TCN":
        enc = TCNEncoder(n_sensors=n_sensors, hidden=64, n_blocks=4, dropout=0.15)
    elif name == "Transformer":
        enc = TransformerBackboneEncoder(n_sensors=n_sensors, hidden=48, n_heads=4, n_layers=2, dropout=0.15)
    else:
        raise ValueError(f"Unknown baseline model name: {name}")
    return MILBaselineClassifier(enc, dropout=0.20)


In [7]:
# ============================================================
# 6. Losses and training utilities
# ============================================================

@dataclass
class TrainCfg:
    lr: float = LR
    weight_decay: float = WEIGHT_DECAY
    batch_size: int = BAG_BATCH_SIZE
    max_epochs: int = MAX_EPOCHS
    patience: int = PATIENCE
    clip_grad: float = 1.0
    seed: int = SEED
    gamma_focal: float = 1.5
    lambda_proto: float = 0.08
    lambda_sparse: float = 0.002
    lambda_orth: float = 0.006
    lambda_effect_l1: float = 0.001
    lambda_attention_stability: float = 0.004
    # Auxiliary consistency and distillation terms.
    lambda_consistency: float = 0.12
    lambda_distill: float = 0.15
    distill_temperature: float = 2.0
    # New robustness/stability terms
    lambda_group_dro: float = 0.70
    dro_eta: float = 3.0
    dro_use_ema: bool = True
    lambda_anchor_aux: float = 0.18
    lambda_static_aux: float = 0.20
    # Regularize the model toward physiologically defined anchors.
    lambda_anchor_relevance: float = 0.14
    min_anchor_relevance_mass: float = 0.55


def make_loader(dataset, batch_size, shuffle):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=False,
    )


def concept_orthogonality_loss(model):
    W = model.relevance_head.weight
    W = F.normalize(W, dim=1)
    G = W @ W.t()
    I = torch.eye(G.shape[0], device=G.device)
    return ((G - I) ** 2).mean()


def focal_bce_with_logits_per_sample(logits, y, pos_weight=None, gamma=1.5):
    bce = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight, reduction="none")
    p = torch.sigmoid(logits)
    pt = p * y + (1.0 - p) * (1.0 - y)
    return ((1.0 - pt).clamp_min(1e-4) ** gamma) * bce


class GroupDROTracker:
    """Track EMA-smoothed group losses and adversarial weights."""
    def __init__(self, n_groups: int, momentum: float = 0.90, device=None):
        self.n_groups = n_groups
        self.momentum = momentum
        self.running_loss = torch.zeros(n_groups, device=device)
        self.seen = torch.zeros(n_groups, dtype=torch.bool, device=device)

    def update_and_get_weights(self, batch_group_losses: dict, eta: float):
        """batch_group_losses: {group_id(int): loss(tensor scalar, detached)}"""
        for g, val in batch_group_losses.items():
            val = val.detach()
            if not self.seen[g]:
                self.running_loss[g] = val
                self.seen[g] = True
            else:
                self.running_loss[g] = self.momentum * self.running_loss[g] + (1.0 - self.momentum) * val
        present = sorted(batch_group_losses.keys())
        present_running = self.running_loss[present]
        weights = torch.softmax(eta * (present_running - present_running.mean()), dim=0)
        return {g: weights[i] for i, g in enumerate(present)}


def group_dro_reduce(per_sample_loss, group_id, tracker: "GroupDROTracker",
                      eta: float = 3.0, dro_weight: float = 0.70, use_ema: bool = True):
    """Blend average risk with worst-group risk.

    use_ema=True (default): weights come from the persistent EMA tracker
    (the fix). use_ema=False reproduces the original v6 behavior for the
    ablation study -- weights recomputed from scratch on each noisy
    mini-batch. The per-sample losses used for backprop are always the
    current batch's values either way, so gradients stay unbiased; only the
    mixture weights differ between the two modes.
    """
    per_sample_loss = per_sample_loss.view(-1)
    group_id = group_id.view(-1)
    mean_loss = per_sample_loss.mean()
    unique_groups = torch.unique(group_id).tolist()
    if len(unique_groups) <= 1:
        return mean_loss, mean_loss.detach(), mean_loss.detach()

    batch_group_losses = {}
    for g in unique_groups:
        mask = group_id == g
        batch_group_losses[g] = per_sample_loss[mask].mean()

    if use_ema:
        weights = tracker.update_and_get_weights(batch_group_losses, eta=eta)
    else:
        tracker.update_and_get_weights(batch_group_losses, eta=eta)  # keep EMA updated for logging only
        present = sorted(batch_group_losses.keys())
        vals = torch.stack([batch_group_losses[g] for g in present])
        w = torch.softmax(eta * (vals.detach() - vals.detach().mean()), dim=0)
        weights = {g: w[i] for i, g in enumerate(present)}

    dro_loss = sum(weights[g] * batch_group_losses[g] for g in unique_groups)
    worst_group_loss = max(batch_group_losses.values()).detach()
    return (1.0 - dro_weight) * mean_loss + dro_weight * dro_loss, dro_loss.detach(), worst_group_loss


def dro_pas_mil_cbm_base_loss(out, y, group_id, model, cfg: TrainCfg, dro_tracker: "GroupDROTracker",
                               pos_weight=None, proto_margin=0.25):
    logits = out["logit"].squeeze(1)
    focal_i = focal_bce_with_logits_per_sample(logits, y, pos_weight=pos_weight, gamma=cfg.gamma_focal)

    dists = out["prototype_dists"]
    y_long = y.long()
    own = dists.gather(1, y_long.view(-1, 1)).squeeze(1)
    other = dists.gather(1, (1 - y_long).view(-1, 1)).squeeze(1)
    proto_i = own + F.relu(proto_margin + own - other)

    anchor_aux_i = F.binary_cross_entropy_with_logits(
        out["anchor_logit"].squeeze(1), y, pos_weight=pos_weight, reduction="none"
    )

    if getattr(model, "use_static", True):
        static_aux_i = F.binary_cross_entropy_with_logits(
            out["static_logit"].squeeze(1), y, pos_weight=pos_weight, reduction="none"
        )
    else:
        static_aux_i = torch.zeros_like(focal_i)

    per_sample = (
        focal_i
        + cfg.lambda_proto * proto_i
        + cfg.lambda_anchor_aux * anchor_aux_i
        + cfg.lambda_static_aux * static_aux_i
    )
    robust_loss, dro_loss, worst_group_loss = group_dro_reduce(
        per_sample, group_id, dro_tracker, eta=cfg.dro_eta, dro_weight=cfg.lambda_group_dro,
        use_ema=cfg.dro_use_ema,
    )

    relevance = out["relevance"].clamp_min(1e-8)
    rel_entropy = -(relevance * relevance.log()).sum(dim=1).mean()
    orth = concept_orthogonality_loss(model)
    effect_l1 = model.concept_effect.abs().mean()

    # Encourage attention to use more than one local window.
    attn = out["attention"].clamp_min(1e-8)
    norm_entropy = -(attn * attn.log()).sum(dim=1) / math.log(max(2, attn.shape[1]))
    attention_stability = (1.0 - norm_entropy).mean()

    # Force at least some mass on physiological anchors, avoiding collapse into learned concepts only.
    anchor_mass = relevance[:, :N_ANCHOR_CONCEPTS].sum(dim=1)
    anchor_relevance_penalty = F.relu(cfg.min_anchor_relevance_mass - anchor_mass).pow(2).mean()

    total = (
        robust_loss
        + cfg.lambda_sparse * rel_entropy
        + cfg.lambda_orth * orth
        + cfg.lambda_effect_l1 * effect_l1
        + cfg.lambda_attention_stability * attention_stability
        + cfg.lambda_anchor_relevance * anchor_relevance_penalty
    )
    return total, {
        "robust": float(robust_loss.detach().cpu()),
        "dro": float(dro_loss.cpu()),
        "worst_group": float(worst_group_loss.cpu()),
        "rel_entropy": float(rel_entropy.detach().cpu()),
        "orth": float(orth.detach().cpu()),
        "anchor_mass": float(anchor_mass.mean().detach().cpu()),
        "anchor_rel_penalty": float(anchor_relevance_penalty.detach().cpu()),
    }


def augment_bag_tensor(x, noise_std=0.015, sensor_drop_prob=0.05, time_mask_prob=0.10):
    x2 = x + torch.randn_like(x) * noise_std
    B, N, T, C = x2.shape
    if sensor_drop_prob > 0:
        mask = (torch.rand(B, 1, 1, C, device=x2.device) < sensor_drop_prob).float()
        x2 = x2 * (1.0 - mask)
    if time_mask_prob > 0:
        for b in range(B):
            if torch.rand((), device=x2.device) < time_mask_prob:
                w = int(torch.randint(max(2, T // 16), max(3, T // 6), (1,), device=x2.device).item())
                s = int(torch.randint(0, max(1, T - w), (1,), device=x2.device).item())
                x2[b, :, s:s + w, :] = 0.0
    return x2


def distillation_loss(logits, teacher_prob, temperature=2.0):
    z = logits.squeeze(1) / temperature
    q = teacher_prob.clamp(0.02, 0.98)
    return F.binary_cross_entropy_with_logits(z, q) * (temperature ** 2)


@torch.no_grad()
def predict_bags(model, bags: List[Dict], bag_windows: int = EVAL_BAG_WINDOWS, batch_size: int = BAG_BATCH_SIZE,
                 temperature: float = 1.0):
    model.eval()
    ds = GaitBagDataset(bags, bag_windows=bag_windows, train=False)
    loader = make_loader(ds, batch_size, shuffle=False)
    rows, concept_rows = [], []
    Tcal = max(float(temperature), 1e-3)
    for batch in loader:
        xb = batch["x"].to(DEVICE, non_blocking=True)
        sb = batch["static"].to(DEVICE, non_blocking=True)
        out = model(xb, sb)
        raw_logits = out["logit"].squeeze(1).cpu().numpy()
        prob = torch.sigmoid(out["logit"].squeeze(1) / Tcal).cpu().numpy()
        reliability = out["reliability"].cpu().numpy()
        concepts = out["bag_concepts"].cpu().numpy()
        relevance = out["relevance"].cpu().numpy()
        mix = out["mix"].detach().cpu().numpy()
        for i in range(len(prob)):
            rows.append({
                "session_id": batch["session_id"][i],
                "person_id": batch["person_id"][i],
                "label": int(batch["y"][i].item()),
                "logit_raw": float(raw_logits[i]),
                "prob": float(prob[i]),
                "reliability": float(reliability[i]),
                "mix_cbm": float(mix[0]),
                "mix_proto": float(mix[1]),
                "mix_anchor": float(mix[2]),
                "mix_static": float(mix[3]),
            })
            cr = {"session_id": batch["session_id"][i], "person_id": batch["person_id"][i], "label": int(batch["y"][i].item())}
            for k, name in enumerate(CONCEPT_NAMES):
                cr[f"{name}_value"] = float(concepts[i, k])
                cr[f"{name}_relevance"] = float(relevance[i, k])
            concept_rows.append(cr)
    return pd.DataFrame(rows), pd.DataFrame(concept_rows)


def fit_temperature_from_logits(y_true, raw_logits, max_iter: int = 80):
    y = np.asarray(y_true).astype(np.float32)
    z = np.asarray(raw_logits).astype(np.float32)
    if len(np.unique(y)) < 2 or len(y) < 4:
        return 1.0
    y_t = torch.tensor(y, dtype=torch.float32)
    z_t = torch.tensor(z, dtype=torch.float32)
    log_temp = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_temp], lr=0.05, max_iter=max_iter, line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        temp = torch.exp(log_temp).clamp(0.20, 10.0)
        loss = F.binary_cross_entropy_with_logits(z_t / temp, y_t)
        loss.backward()
        return loss

    try:
        opt.step(closure)
        temp = float(torch.exp(log_temp).detach().clamp(0.20, 10.0).item())
    except Exception:
        temp = 1.0
    return temp


def train_dro_pas_mil_cbm(train_bags, val_bags, cfg: TrainCfg, fold: int = 0, seed_offset: int = 0,
                           model_overrides: Optional[dict] = None):
    seed_everything(cfg.seed + fold + seed_offset)
    static_dim = train_bags[0]["static_feat"].shape[0]
    model_kwargs = dict(
        n_sensors=N_SENSORS, hidden=HIDDEN, n_learned=N_LEARNED_CONCEPTS, dropout=0.15,
        n_prototypes_per_class=N_PROTOTYPES_PER_CLASS, static_feat_dim=static_dim, use_static=True,
    )
    if model_overrides:
        model_kwargs.update(model_overrides)
    model = DROMultiPrototypePASMILCBM(**model_kwargs).to(DEVICE)
    dro_tracker = GroupDROTracker(n_groups=DRO_GROUPS, momentum=DRO_EMA_MOMENTUM, device=DEVICE)

    ytr = np.array([b["label"] for b in train_bags], dtype=int)
    n_pos = max(1, int(np.sum(ytr == 1)))
    n_neg = max(1, int(np.sum(ytr == 0)))
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=3)
    train_ds = GaitBagDataset(train_bags, bag_windows=TRAIN_BAG_WINDOWS, train=True)
    train_loader = make_loader(train_ds, cfg.batch_size, shuffle=True)

    best_state, best_score, bad = None, -np.inf, 0
    history = []
    t0 = time.time()

    for epoch in range(1, cfg.max_epochs + 1):
        model.train()
        losses = []
        anchor_masses = []
        worst_losses = []
        for batch in train_loader:
            xb = batch["x"].to(DEVICE, non_blocking=True)
            sb = batch["static"].to(DEVICE, non_blocking=True)
            yb = batch["y"].to(DEVICE, non_blocking=True)
            tb = batch["teacher"].to(DEVICE, non_blocking=True)
            gb = batch["group"].to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            out1 = model(xb, sb)
            x_aug = augment_bag_tensor(xb)
            out2 = model(x_aug, sb)

            loss1, parts1 = dro_pas_mil_cbm_base_loss(out1, yb, gb, model, cfg, dro_tracker, pos_weight=pos_weight)
            loss2, parts2 = dro_pas_mil_cbm_base_loss(out2, yb, gb, model, cfg, dro_tracker, pos_weight=pos_weight)
            p1 = torch.sigmoid(out1["logit"].squeeze(1))
            p2 = torch.sigmoid(out2["logit"].squeeze(1))
            consistency = F.mse_loss(p1, p2) + 0.05 * F.mse_loss(out1["bag_concepts"], out2["bag_concepts"])
            distill = 0.5 * (
                distillation_loss(out1["logit"], tb, cfg.distill_temperature)
                + distillation_loss(out2["logit"], tb, cfg.distill_temperature)
            )
            loss = 0.5 * (loss1 + loss2) + cfg.lambda_consistency * consistency + cfg.lambda_distill * distill
            loss.backward()
            if cfg.clip_grad is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad)
            opt.step()
            losses.append(float(loss.detach().cpu()))
            anchor_masses.append(0.5 * (parts1["anchor_mass"] + parts2["anchor_mass"]))
            worst_losses.append(0.5 * (parts1["worst_group"] + parts2["worst_group"]))

        val_pred_raw, _ = predict_bags(model, val_bags, EVAL_BAG_WINDOWS, cfg.batch_size, temperature=1.0)
        val_temperature = fit_temperature_from_logits(val_pred_raw["label"].values, val_pred_raw["logit_raw"].values)
        val_pred_df, _ = predict_bags(model, val_bags, EVAL_BAG_WINDOWS, cfg.batch_size, temperature=val_temperature)
        val_thr = threshold_youden_with_sensitivity_floor(
            val_pred_df["label"].values, val_pred_df["prob"].values, MIN_VALIDATION_SENSITIVITY
        )
        val_metrics = compute_metrics_with_threshold(val_pred_df["label"], val_pred_df["prob"], val_thr, prefix="val_")
        val_auc = val_metrics["val_AUC"]
        val_f1 = val_metrics["val_F1"]
        val_sens = val_metrics["val_Sens"]
        val_spec = val_metrics["val_Spec"]
        score = (val_auc if np.isfinite(val_auc) else 0.0) + 0.15 * val_f1 + 0.10 * min(val_sens, val_spec)
        scheduler.step(score)
        train_loss = float(np.nanmean(losses)) if losses else np.nan
        history.append({
            "fold": fold,
            "seed": cfg.seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_thr": val_thr,
            "val_temperature": val_temperature,
            "val_auc": val_auc,
            "val_f1": val_f1,
            "val_sens": val_sens,
            "val_spec": val_spec,
            "anchor_mass": float(np.nanmean(anchor_masses)) if anchor_masses else np.nan,
            "worst_group_loss": float(np.nanmean(worst_losses)) if worst_losses else np.nan,
            "score": score,
        })

        if PRINT_EVERY_EPOCH:
            elapsed = time.time() - t0
            print(
                f"    seed {cfg.seed} epoch {epoch:03d} | loss={train_loss:.4f} | val_auc={val_auc:.3f} | "
                f"val_f1={val_f1:.3f} | sens={val_sens:.3f} | spec={val_spec:.3f} | "
                f"T={val_temperature:.2f} | thr={val_thr:.3f} | anchor_mass={np.nanmean(anchor_masses):.3f} | "
                f"bad={bad}/{cfg.patience} | elapsed={elapsed:.1f}s",
                flush=True,
            )

        if score > best_score + 1e-4:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg.patience:
                print(f"    early stopping at epoch {epoch}; best_score={best_score:.4f}", flush=True)
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    # Final temperature from the restored best model.
    val_pred_raw, _ = predict_bags(model, val_bags, EVAL_BAG_WINDOWS, cfg.batch_size, temperature=1.0)
    temperature = fit_temperature_from_logits(val_pred_raw["label"].values, val_pred_raw["logit_raw"].values)
    return model, pd.DataFrame(history), float(temperature)

# ============================================================
# 6b. Generic training/prediction for the SOTA deep-learning baselines
# ============================================================
# Baselines use focal loss with the same MIL, calibration, and thresholding protocol.

@dataclass
class BaselineTrainCfg:
    lr: float = 1e-3
    weight_decay: float = 1e-4
    batch_size: int = BAG_BATCH_SIZE
    max_epochs: int = MAX_EPOCHS_BASELINE
    patience: int = PATIENCE_BASELINE
    clip_grad: float = 1.0
    seed: int = BASELINE_SEED
    gamma_focal: float = 1.5


@torch.no_grad()
def predict_bags_baseline(model, bags: List[Dict], bag_windows: int = EVAL_BAG_WINDOWS,
                           batch_size: int = BAG_BATCH_SIZE, temperature: float = 1.0):
    model.eval()
    ds = GaitBagDataset(bags, bag_windows=bag_windows, train=False)
    loader = make_loader(ds, batch_size, shuffle=False)
    rows = []
    Tcal = max(float(temperature), 1e-3)
    for batch in loader:
        xb = batch["x"].to(DEVICE, non_blocking=True)
        out = model(xb)
        raw_logits = out["logit"].squeeze(1).cpu().numpy()
        prob = torch.sigmoid(out["logit"].squeeze(1) / Tcal).cpu().numpy()
        for i in range(len(prob)):
            rows.append({
                "session_id": batch["session_id"][i],
                "person_id": batch["person_id"][i],
                "label": int(batch["y"][i].item()),
                "logit_raw": float(raw_logits[i]),
                "prob": float(prob[i]),
            })
    return pd.DataFrame(rows)


def train_mil_baseline(model_name: str, train_bags, val_bags, cfg: BaselineTrainCfg, fold: int = 0):
    seed_everything(cfg.seed + fold)
    model = build_baseline_model(model_name, n_sensors=N_SENSORS).to(DEVICE)

    ytr = np.array([b["label"] for b in train_bags], dtype=int)
    n_pos = max(1, int(np.sum(ytr == 1)))
    n_neg = max(1, int(np.sum(ytr == 0)))
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=3)
    train_ds = GaitBagDataset(train_bags, bag_windows=TRAIN_BAG_WINDOWS, train=True)
    train_loader = make_loader(train_ds, cfg.batch_size, shuffle=True)

    best_state, best_score, bad = None, -np.inf, 0
    t0 = time.time()
    for epoch in range(1, cfg.max_epochs + 1):
        model.train()
        for batch in train_loader:
            xb = batch["x"].to(DEVICE, non_blocking=True)
            yb = batch["y"].to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            out = model(xb)
            logits = out["logit"].squeeze(1)
            loss = focal_bce_with_logits_per_sample(logits, yb, pos_weight=pos_weight, gamma=cfg.gamma_focal).mean()
            loss.backward()
            if cfg.clip_grad is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad)
            opt.step()

        val_pred_raw = predict_bags_baseline(model, val_bags, EVAL_BAG_WINDOWS, cfg.batch_size, temperature=1.0)
        temperature = fit_temperature_from_logits(val_pred_raw["label"].values, val_pred_raw["logit_raw"].values)
        val_pred_df = predict_bags_baseline(model, val_bags, EVAL_BAG_WINDOWS, cfg.batch_size, temperature=temperature)
        val_thr = threshold_youden_with_sensitivity_floor(
            val_pred_df["label"].values, val_pred_df["prob"].values, MIN_VALIDATION_SENSITIVITY
        )
        val_metrics = compute_metrics_with_threshold(val_pred_df["label"], val_pred_df["prob"], val_thr, prefix="val_")
        val_auc = val_metrics["val_AUC"]
        val_f1 = val_metrics["val_F1"]
        val_sens = val_metrics["val_Sens"]
        val_spec = val_metrics["val_Spec"]
        score = (val_auc if np.isfinite(val_auc) else 0.0) + 0.15 * val_f1 + 0.10 * min(val_sens, val_spec)
        scheduler.step(score)

        if PRINT_EVERY_EPOCH:
            elapsed = time.time() - t0
            print(
                f"    [{model_name}] epoch {epoch:03d} | val_auc={val_auc:.3f} | val_f1={val_f1:.3f} | "
                f"sens={val_sens:.3f} | spec={val_spec:.3f} | bad={bad}/{cfg.patience} | elapsed={elapsed:.1f}s",
                flush=True,
            )

        if score > best_score + 1e-4:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg.patience:
                print(f"    [{model_name}] early stopping at epoch {epoch}; best_score={best_score:.4f}", flush=True)
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    val_pred_raw = predict_bags_baseline(model, val_bags, EVAL_BAG_WINDOWS, cfg.batch_size, temperature=1.0)
    temperature = fit_temperature_from_logits(val_pred_raw["label"].values, val_pred_raw["logit_raw"].values)
    return model, float(temperature)


In [8]:
# ============================================================
# 7. Run experiment
# ============================================================

cfg = TrainCfg()
all_rows = []
fold_histories = []
session_pred_rows = []
concept_rows_all = []

# Per-baseline-model accumulators (session_id -> ensemble prediction rows per fold).
baseline_session_pred_rows = {name: [] for name in BASELINE_MODEL_NAMES} if RUN_BASELINES else {}

print("\nRunning DRO-PAS-MIL-CBM (+ SOTA baselines) grouped CV...")
for fold, (tr_idx, te_idx) in enumerate(SPLITS, 1):
    print("\n" + "=" * 70)
    print(f"Fold {fold}/{len(SPLITS)}")

    train_records_full = get_records_by_indices(records, tr_idx)
    test_records = get_records_by_indices(records, te_idx)
    train_records, val_records = train_val_split_by_person(train_records_full, VAL_PERSON_RATIO, SEED + fold)
    scaler = fit_scaler(train_records)

    # Strong static teacher is fitted only on training persons of this fold.
    teacher = fit_static_teacher(train_records)
    # Scaler for the static fusion branch, fit on TRAIN persons of this fold only
    # (same anti-leakage discipline as fit_scaler for the raw sensor scaler).
    static_scaler = fit_static_scaler(train_records)

    train_bags = build_bags(train_records, scaler, teacher_model=teacher,
                             max_pool_windows=MAX_POOL_WINDOWS_PER_SESSION, static_scaler=static_scaler)
    val_bags = build_bags(val_records, scaler, teacher_model=None,
                           max_pool_windows=MAX_POOL_WINDOWS_PER_SESSION, static_scaler=static_scaler)
    test_bags = build_bags(test_records, scaler, teacher_model=None,
                            max_pool_windows=MAX_POOL_WINDOWS_PER_SESSION, static_scaler=static_scaler)

    print(
        f"Bags: train={len(train_bags)}, val={len(val_bags)}, test={len(test_bags)} | "
        f"train PD={np.mean([b['label'] for b in train_bags]):.3f}, "
        f"val PD={np.mean([b['label'] for b in val_bags]):.3f}, "
        f"test PD={np.mean([b['label'] for b in test_bags]):.3f}"
    )

    # ---------------- Proposed model: DRO-PAS-MIL-CBM ensemble ----------------
    seed_val_preds = []
    seed_test_preds = []
    seed_concepts = []
    fold_temperatures = []

    for s_i, seed in enumerate(ENSEMBLE_SEEDS):
        print(f"\n  Training ensemble seed {s_i+1}/{len(ENSEMBLE_SEEDS)}: {seed}")
        cfg_seed = replace(cfg, seed=int(seed))
        model, hist, temperature = train_dro_pas_mil_cbm(train_bags, val_bags, cfg_seed, fold=fold, seed_offset=1000 * s_i)
        hist["ensemble_seed"] = seed
        fold_histories.append(hist)
        fold_temperatures.append(temperature)

        val_pred_df, _ = predict_bags(model, val_bags, EVAL_BAG_WINDOWS, cfg.batch_size, temperature=temperature)
        test_pred_df, concept_df = predict_bags(model, test_bags, EVAL_BAG_WINDOWS, cfg.batch_size, temperature=temperature)
        val_pred_df["ensemble_seed"] = seed
        test_pred_df["ensemble_seed"] = seed
        concept_df["ensemble_seed"] = seed
        seed_val_preds.append(val_pred_df)
        seed_test_preds.append(test_pred_df)
        seed_concepts.append(concept_df)

    val_all = pd.concat(seed_val_preds, ignore_index=True)
    test_all = pd.concat(seed_test_preds, ignore_index=True)

    val_ens = val_all.groupby(["session_id", "person_id", "label"], as_index=False).agg(
        prob=("prob", "mean"),
        logit_raw=("logit_raw", "mean"),
        reliability=("reliability", "mean"),
        mix_cbm=("mix_cbm", "mean"),
        mix_proto=("mix_proto", "mean"),
        mix_anchor=("mix_anchor", "mean"),
        mix_static=("mix_static", "mean"),
    )
    test_ens = test_all.groupby(["session_id", "person_id", "label"], as_index=False).agg(
        prob=("prob", "mean"),
        logit_raw=("logit_raw", "mean"),
        reliability=("reliability", "mean"),
        mix_cbm=("mix_cbm", "mean"),
        mix_proto=("mix_proto", "mean"),
        mix_anchor=("mix_anchor", "mean"),
        mix_static=("mix_static", "mean"),
    )

    thr = threshold_youden_with_sensitivity_floor(
        val_ens["label"].values, val_ens["prob"].values, MIN_VALIDATION_SENSITIVITY
    )
    test_ens["fold"] = fold
    test_ens["thr_fold"] = thr
    test_ens["temperature_mean"] = float(np.mean(fold_temperatures))
    test_ens["pred"] = (test_ens["prob"] >= thr).astype(int)

    concept_all = pd.concat(seed_concepts, ignore_index=True)
    concept_numeric_cols = [c for c in concept_all.columns if c.endswith("_value") or c.endswith("_relevance")]
    concept_df = concept_all.groupby(["session_id", "person_id", "label"], as_index=False)[concept_numeric_cols].mean()
    concept_df["fold"] = fold

    fold_metrics = compute_metrics_with_predictions(
        test_ens["label"], test_ens["prob"], test_ens["pred"], prefix="Session_"
    )
    print(
        f"Fold {fold} DRO-PAS-MIL-CBM session AUC={fold_metrics['Session_AUC']:.3f}, "
        f"F1={fold_metrics['Session_F1']:.3f}, sens={fold_metrics['Session_Sens']:.3f}, "
        f"spec={fold_metrics['Session_Spec']:.3f}, thr={thr:.3f}, Tmean={np.mean(fold_temperatures):.2f}",
        flush=True,
    )

    session_pred_rows.append(test_ens)
    concept_rows_all.append(concept_df)

    # ---------------- SOTA deep-learning baselines (same bags, same fold) ----------------
    if RUN_BASELINES:
        base_cfg = BaselineTrainCfg()
        for bname in BASELINE_MODEL_NAMES:
            print(f"\n  Training baseline {bname} (fold {fold})")
            bmodel, btemp = train_mil_baseline(bname, train_bags, val_bags, base_cfg, fold=fold)
            bval = predict_bags_baseline(bmodel, val_bags, EVAL_BAG_WINDOWS, base_cfg.batch_size, temperature=btemp)
            btest = predict_bags_baseline(bmodel, test_bags, EVAL_BAG_WINDOWS, base_cfg.batch_size, temperature=btemp)
            bthr = threshold_youden_with_sensitivity_floor(
                bval["label"].values, bval["prob"].values, MIN_VALIDATION_SENSITIVITY
            )
            btest["fold"] = fold
            btest["thr_fold"] = bthr
            btest["pred"] = (btest["prob"] >= bthr).astype(int)
            bmetrics = compute_metrics_with_predictions(btest["label"], btest["prob"], btest["pred"], prefix="Session_")
            print(
                f"    Fold {fold} {bname} session AUC={bmetrics['Session_AUC']:.3f}, "
                f"F1={bmetrics['Session_F1']:.3f}, thr={bthr:.3f}", flush=True,
            )
            baseline_session_pred_rows[bname].append(btest)

session_pred_df = pd.concat(session_pred_rows, ignore_index=True)
concept_session_df = pd.concat(concept_rows_all, ignore_index=True)
history_df = pd.concat(fold_histories, ignore_index=True) if fold_histories else pd.DataFrame()

pas_session_metrics = compute_metrics_with_predictions(
    session_pred_df["label"], session_pred_df["prob"], session_pred_df["pred"], prefix="Session_"
)

person_pred_df = session_pred_df.groupby("person_id").agg(
    label=("label", "max"),
    prob=("prob", "mean"),
    thr_fold=("thr_fold", "mean"),
    n_sessions=("session_id", "count"),
    reliability=("reliability", "mean"),
    mix_cbm=("mix_cbm", "mean"),
    mix_proto=("mix_proto", "mean"),
    mix_anchor=("mix_anchor", "mean"),
    mix_static=("mix_static", "mean"),
).reset_index()
person_pred_df["pred"] = (person_pred_df["prob"] >= person_pred_df["thr_fold"]).astype(int)
pas_person_metrics = compute_metrics_with_predictions(
    person_pred_df["label"], person_pred_df["prob"], person_pred_df["pred"], prefix="Person_"
)

all_rows.append({"Model": "DRO-PAS-MIL-CBM_multi_proto_calibrated_ensemble", **pas_session_metrics, **pas_person_metrics})

# ---------------- Aggregate baseline results (session + person level) ----------------
baseline_session_dfs = {}
if RUN_BASELINES:
    for bname in BASELINE_MODEL_NAMES:
        bdf = pd.concat(baseline_session_pred_rows[bname], ignore_index=True)
        baseline_session_dfs[bname] = bdf
        b_sess_metrics = compute_metrics_with_predictions(bdf["label"], bdf["prob"], bdf["pred"], prefix="Session_")
        b_person_df = bdf.groupby("person_id").agg(
            label=("label", "max"), prob=("prob", "mean"), thr_fold=("thr_fold", "mean"),
        ).reset_index()
        b_person_df["pred"] = (b_person_df["prob"] >= b_person_df["thr_fold"]).astype(int)
        b_person_metrics = compute_metrics_with_predictions(b_person_df["label"], b_person_df["prob"], b_person_df["pred"], prefix="Person_")
        all_rows.append({"Model": bname, **b_sess_metrics, **b_person_metrics})

results_df = pd.DataFrame(all_rows)

print("\nFinal results")
display(results_df)

# Concept ranking: average relevance plus class shift in the session-level concept values.
concept_rank_rows = []
for k, name in enumerate(CONCEPT_NAMES):
    v_col = f"{name}_value"
    r_col = f"{name}_relevance"
    ctrl = concept_session_df.loc[concept_session_df["label"] == 0, v_col]
    pdv = concept_session_df.loc[concept_session_df["label"] == 1, v_col]
    concept_rank_rows.append({
        "concept": name,
        "mean_relevance": float(concept_session_df[r_col].mean()),
        "control_mean": float(ctrl.mean()) if len(ctrl) else np.nan,
        "pd_mean": float(pdv.mean()) if len(pdv) else np.nan,
        "pd_minus_control": float(pdv.mean() - ctrl.mean()) if len(ctrl) and len(pdv) else np.nan,
        "abs_class_shift": float(abs(pdv.mean() - ctrl.mean())) if len(ctrl) and len(pdv) else np.nan,
    })
concept_rank = pd.DataFrame(concept_rank_rows).sort_values(
    ["mean_relevance", "abs_class_shift"], ascending=False
)
print("\nConcept ranking")
display(concept_rank)

# Fold stability summary (proposed model)
fold_stability = session_pred_df.groupby("fold").apply(
    lambda g: pd.Series(compute_metrics_with_predictions(g["label"], g["prob"], g["pred"], prefix=""))
).reset_index()
print("\nFold stability (DRO-PAS-MIL-CBM)")
display(fold_stability[["fold", "AUC", "Acc", "F1", "Sens", "Spec", "TP", "FN", "TN", "FP"]])
fold_auc_vals = fold_stability["AUC"].dropna().values
print(
    f"\nFold AUC summary: mean={np.mean(fold_auc_vals):.3f}, std={np.std(fold_auc_vals):.3f}, "
    f"min={np.min(fold_auc_vals):.3f}, max={np.max(fold_auc_vals):.3f}, range={np.ptp(fold_auc_vals):.3f}"
)
print(
    "This std/range is the honest measure of the method's stability -- a low mean fold AUC "
    "std/range is what 'stability-enhanced' should actually demonstrate, not just the pooled AUC."
)

print("\nAverage prediction mixture weights")
print(session_pred_df[["mix_cbm", "mix_proto", "mix_anchor", "mix_static"]].mean())

# Fold-level stability for each baseline too, for a fair "stability" comparison.
if RUN_BASELINES:
    print("\nFold AUC stability, baselines vs proposed:")
    stability_rows = [{
        "Model": "DRO-PAS-MIL-CBM", "fold_auc_mean": float(np.mean(fold_auc_vals)),
        "fold_auc_std": float(np.std(fold_auc_vals)), "fold_auc_range": float(np.ptp(fold_auc_vals)),
    }]
    for bname in BASELINE_MODEL_NAMES:
        bdf = baseline_session_dfs[bname]
        b_fold_auc = bdf.groupby("fold").apply(lambda g: safe_auc(g["label"], g["prob"])).dropna().values
        stability_rows.append({
            "Model": bname, "fold_auc_mean": float(np.mean(b_fold_auc)),
            "fold_auc_std": float(np.std(b_fold_auc)), "fold_auc_range": float(np.ptp(b_fold_auc)),
        })
    stability_df = pd.DataFrame(stability_rows)
    display(stability_df)

# --------------------------------------------------------------------------
# Statistical comparison: proposed model vs. each SOTA baseline
# (session-level AUC, person-clustered paired bootstrap). Answers "is the
# proposed method actually better than each baseline, or within noise?"
# rather than relying on single point estimates.
# --------------------------------------------------------------------------
sig_tests = {}
if RUN_BASELINES:
    for bname in BASELINE_MODEL_NAMES:
        bdf = baseline_session_dfs[bname][["session_id", "prob"]].rename(columns={"prob": "base_prob"})
        compare_df = session_pred_df.merge(bdf, on="session_id", how="inner")
        st = paired_bootstrap_auc_test(
            compare_df["label"].values,
            compare_df["prob"].values,       # DRO-PAS-MIL-CBM
            compare_df["base_prob"].values,  # baseline
            compare_df["person_id"].values,
            n_boot=2000,
            seed=SEED,
        )
        sig_tests[bname] = st
        verdict = (
            "proposed significantly better" if st["ci95_lo"] > 0 else
            f"{bname} significantly better" if st["ci95_hi"] < 0 else
            "not statistically significant"
        )
        print(
            f"\nDRO-PAS-MIL-CBM vs {bname} (session AUC, person-clustered bootstrap, n=2000): "
            f"diff={st['auc_diff']:+.4f}, 95% CI [{st['ci95_lo']:+.4f}, {st['ci95_hi']:+.4f}], "
            f"p={st['p_value']:.4f} -> {verdict}"
        )



Running DRO-PAS-MIL-CBM (+ SOTA baselines) grouped CV...

Fold 1/5
  Bags: sessions=185, mean_pool_windows=83.4, min=34, max=96
  Bags: sessions=56, mean_pool_windows=76.4, min=31, max=96
  Bags: sessions=65, mean_pool_windows=86.2, min=31, max=96
Bags: train=185, val=56, test=65 | train PD=0.681, val PD=0.750, test PD=0.708

  Training ensemble seed 1/3: 42
    seed 42 epoch 001 | loss=0.8884 | val_auc=0.808 | val_f1=0.857 | sens=0.786 | spec=0.857 | T=0.20 | thr=0.577 | anchor_mass=0.582 | bad=0/12 | elapsed=40.9s
    seed 42 epoch 002 | loss=0.8466 | val_auc=0.844 | val_f1=0.872 | sens=0.810 | spec=0.857 | T=0.20 | thr=0.511 | anchor_mass=0.606 | bad=0/12 | elapsed=75.9s
    seed 42 epoch 003 | loss=0.8012 | val_auc=0.852 | val_f1=0.916 | sens=0.905 | spec=0.786 | T=0.20 | thr=0.391 | anchor_mass=0.634 | bad=0/12 | elapsed=110.5s
    seed 42 epoch 004 | loss=0.7523 | val_auc=0.855 | val_f1=0.918 | sens=0.929 | spec=0.714 | T=0.46 | thr=0.393 | anchor_mass=0.638 | bad=0/12 | elapsed

,Model,Session_AUC,Session_AP,Session_Acc,Session_Prec,Session_Rec,Session_F1,Session_Spec,Session_Sens,Session_TN,...,Person_Acc,Person_Prec,Person_Rec,Person_F1,Person_Spec,Person_Sens,Person_TN,Person_FP,Person_FN,Person_TP
0,DRO-PAS-MIL-CBM_multi_proto_calibrated_ensemble,0.795815,0.871055,0.709150,0.853107,0.705607,0.772379,0.717391,0.705607,66,...,0.696970,0.765432,0.666667,0.712644,0.736111,0.666667,53,19,31,62
1,CNN1D,0.711703,0.846491,0.624183,0.796407,0.621495,0.698163,0.630435,0.621495,58,...,0.630303,0.690476,0.623656,0.655367,0.638889,0.623656,46,26,35,58
2,ResNet1D,0.690725,0.820901,0.676471,0.814208,0.696262,0.750630,0.630435,0.696262,58,...,0.703030,0.729167,0.752688,0.740741,0.638889,0.752688,46,26,23,70
3,BiLSTM,0.715715,0.830111,0.643791,0.790055,0.668224,0.724051,0.586957,0.668224,54,...,0.660606,0.694737,0.709677,0.702128,0.597222,0.709677,43,29,27,66
4,TCN,0.638054,0.792014,0.656863,0.788360,0.696262,0.739454,0.565217,0.696262,52,...,0.654545,0.687500,0.709677,0.698413,0.583333,0.709677,42,30,27,66
5,Transformer,0.650701,0.790618,0.640523,0.757426,0.714953,0.735577,0.467391,0.714953,43,...,0.642424,0.657407,0.763441,0.706468,0.486111,0.763441,35,37,22,71



Concept ranking


,concept,mean_relevance,control_mean,pd_mean,pd_minus_control,abs_class_shift
7,C08_bilateral_change_mismatch,0.152538,-0.083771,-0.115843,-0.032072,0.032072
4,C05_asymmetry_variability,0.152424,0.916416,0.945224,0.028807,0.028807
3,C04_abs_left_right_asymmetry,0.149756,0.369161,0.364216,-0.004945,0.004945
0,C01_total_load_level,0.107714,-0.487560,-0.436207,0.051353,0.051353
8,C09_learned_temporal_1,0.068132,0.048803,0.098504,0.049701,0.049701
5,C06_signed_left_right_dominance,0.065562,-0.443463,-0.472889,-0.029425,0.029425
10,C11_learned_temporal_3,0.061512,0.013480,0.047641,0.034161,0.034161
13,C14_learned_temporal_6,0.048208,-0.040438,-0.065855,-0.025417,0.025417
14,C15_learned_temporal_7,0.039428,-0.019583,0.008897,0.028481,0.028481
6,C07_force_change_energy,0.030761,-0.442365,-0.460138,-0.017773,0.017773



Fold stability (DRO-PAS-MIL-CBM)


,fold,AUC,Acc,F1,Sens,Spec,TP,FN,TN,FP
0,1,0.844394,0.800000,0.857143,0.847826,0.684211,39.0,7.0,13.0,6.0
1,2,0.939516,0.893617,0.918033,0.903226,0.875000,28.0,3.0,14.0,2.0
2,3,0.739130,0.672131,0.777778,0.760870,0.400000,35.0,11.0,6.0,9.0
3,4,0.766484,0.616438,0.688889,0.596154,0.666667,31.0,21.0,14.0,7.0
4,5,0.870574,0.616667,0.610169,0.461538,0.904762,18.0,21.0,19.0,2.0



Fold AUC summary: mean=0.832, std=0.072, min=0.739, max=0.940, range=0.200
This std/range is the honest measure of the method's stability -- a low mean fold AUC std/range is what 'stability-enhanced' should actually demonstrate, not just the pooled AUC.

Average prediction mixture weights
mix_cbm       0.243261
mix_proto     0.243423
mix_anchor    0.245252
mix_static    0.268064
dtype: float64

Fold AUC stability, baselines vs proposed:


,Model,fold_auc_mean,fold_auc_std,fold_auc_range
0,DRO-PAS-MIL-CBM,0.832020,0.072275,0.200386
1,CNN1D,0.744848,0.063308,0.183409
2,ResNet1D,0.745309,0.112440,0.314605
3,BiLSTM,0.742612,0.075997,0.188616
4,TCN,0.696411,0.089756,0.225895
5,Transformer,0.647403,0.098847,0.307051



DRO-PAS-MIL-CBM vs CNN1D (session AUC, person-clustered bootstrap, n=2000): diff=+0.0841, 95% CI [-0.0128, +0.1826], p=0.0850 -> not statistically significant

DRO-PAS-MIL-CBM vs ResNet1D (session AUC, person-clustered bootstrap, n=2000): diff=+0.1051, 95% CI [+0.0085, +0.2039], p=0.0320 -> proposed significantly better

DRO-PAS-MIL-CBM vs BiLSTM (session AUC, person-clustered bootstrap, n=2000): diff=+0.0801, 95% CI [+0.0026, +0.1574], p=0.0440 -> proposed significantly better

DRO-PAS-MIL-CBM vs TCN (session AUC, person-clustered bootstrap, n=2000): diff=+0.1578, 95% CI [+0.0778, +0.2409], p=0.0000 -> proposed significantly better

DRO-PAS-MIL-CBM vs Transformer (session AUC, person-clustered bootstrap, n=2000): diff=+0.1451, 95% CI [+0.0558, +0.2454], p=0.0010 -> proposed significantly better


In [ ]:
# ============================================================
# 7b. Ablation study
# ============================================================
# Reduced ablation protocol; bags are reused within each fold.

ABLATION_RESULTS = []

if RUN_ABLATIONS:
    ABLATION_FOLD_IDS = list(range(1, min(ABLATION_N_FOLDS, len(SPLITS)) + 1))
    print(f"Running ablation study on folds {ABLATION_FOLD_IDS} (of {len(SPLITS)} total), "
          f"max_epochs<={ABLATION_MAX_EPOCHS}, patience={ABLATION_PATIENCE}")

    ablation_fold_cache = {}
    for fold in ABLATION_FOLD_IDS:
        tr_idx, te_idx = SPLITS[fold - 1]
        train_records_full = get_records_by_indices(records, tr_idx)
        test_records = get_records_by_indices(records, te_idx)
        train_records, val_records = train_val_split_by_person(train_records_full, VAL_PERSON_RATIO, SEED + fold)
        scaler = fit_scaler(train_records)
        teacher = fit_static_teacher(train_records)
        static_scaler = fit_static_scaler(train_records)
        train_bags = build_bags(train_records, scaler, teacher_model=teacher,
                                 max_pool_windows=MAX_POOL_WINDOWS_PER_SESSION, static_scaler=static_scaler)
        val_bags = build_bags(val_records, scaler, teacher_model=None,
                               max_pool_windows=MAX_POOL_WINDOWS_PER_SESSION, static_scaler=static_scaler)
        test_bags = build_bags(test_records, scaler, teacher_model=None,
                                max_pool_windows=MAX_POOL_WINDOWS_PER_SESSION, static_scaler=static_scaler)
        ablation_fold_cache[fold] = (train_bags, val_bags, test_bags)

    def run_ablation_variant(name, cfg_overrides=None, model_overrides=None, seeds=(42,)):
        cfg_overrides = cfg_overrides or {}
        model_overrides = model_overrides or {}
        variant_cfg = replace(TrainCfg(), max_epochs=ABLATION_MAX_EPOCHS, patience=ABLATION_PATIENCE, **cfg_overrides)
        rows = []
        for fold in ABLATION_FOLD_IDS:
            train_bags, val_bags, test_bags = ablation_fold_cache[fold]
            seed_val_preds, seed_test_preds = [], []
            for s_i, seed in enumerate(seeds):
                cfg_seed = replace(variant_cfg, seed=int(seed))
                model, _, temperature = train_dro_pas_mil_cbm(
                    train_bags, val_bags, cfg_seed, fold=fold, seed_offset=1000 * s_i,
                    model_overrides=model_overrides,
                )
                val_pred_df, _ = predict_bags(model, val_bags, EVAL_BAG_WINDOWS, cfg_seed.batch_size, temperature=temperature)
                test_pred_df, _ = predict_bags(model, test_bags, EVAL_BAG_WINDOWS, cfg_seed.batch_size, temperature=temperature)
                seed_val_preds.append(val_pred_df)
                seed_test_preds.append(test_pred_df)
            val_ens = pd.concat(seed_val_preds, ignore_index=True).groupby(
                ["session_id", "person_id", "label"], as_index=False).agg(prob=("prob", "mean"))
            test_ens = pd.concat(seed_test_preds, ignore_index=True).groupby(
                ["session_id", "person_id", "label"], as_index=False).agg(prob=("prob", "mean"))
            thr = threshold_youden_with_sensitivity_floor(
                val_ens["label"].values, val_ens["prob"].values, MIN_VALIDATION_SENSITIVITY
            )
            test_ens["pred"] = (test_ens["prob"] >= thr).astype(int)
            test_ens["fold"] = fold
            rows.append(test_ens)
        all_test = pd.concat(rows, ignore_index=True)
        m = compute_metrics_with_predictions(all_test["label"], all_test["prob"], all_test["pred"], prefix="")
        fold_aucs = all_test.groupby("fold").apply(lambda g: safe_auc(g["label"], g["prob"])).dropna().values
        return {
            "variant": name,
            "AUC": m["AUC"], "F1": m["F1"], "Sens": m["Sens"], "Spec": m["Spec"],
            "fold_auc_mean": float(np.nanmean(fold_aucs)) if len(fold_aucs) else np.nan,
            "fold_auc_std": float(np.nanstd(fold_aucs)) if len(fold_aucs) else np.nan,
        }

    # "Full model" row: reuse the already-computed cell-8 results restricted
    # to the ablation fold subset instead of re-training (saves one variant's
    # worth of compute, and is the exact same trained models/predictions).
    full_subset = session_pred_df[session_pred_df["fold"].isin(ABLATION_FOLD_IDS)]
    m_full = compute_metrics_with_predictions(full_subset["label"], full_subset["prob"], full_subset["pred"], prefix="")
    fold_aucs_full = full_subset.groupby("fold").apply(lambda g: safe_auc(g["label"], g["prob"])).dropna().values
    ABLATION_RESULTS.append({
        "variant": "full_model (all fixes, 3-seed ensemble)",
        "AUC": m_full["AUC"], "F1": m_full["F1"], "Sens": m_full["Sens"], "Spec": m_full["Spec"],
        "fold_auc_mean": float(np.nanmean(fold_aucs_full)), "fold_auc_std": float(np.nanstd(fold_aucs_full)),
    })

    variant_specs = [
        # (name, TrainCfg overrides, model constructor overrides, seeds)
        ("no_static_fusion", {}, {"use_static": False}, (42,)),
        ("no_ema_dro (vanilla per-batch DRO, v6 behavior)", {"dro_use_ema": False}, {}, (42,)),
        ("no_group_dro (DRO off entirely)", {"lambda_group_dro": 0.0}, {}, (42,)),
        ("single_prototype (was 3/class)", {}, {"n_prototypes_per_class": 1}, (42,)),
        ("weak_anchor_floor (v6 settings: 0.35/0.06)", {"min_anchor_relevance_mass": 0.35, "lambda_anchor_relevance": 0.06}, {}, (42,)),
        ("heavy_aux_losses (v6 settings: consistency=0.20, distill=0.22)", {"lambda_consistency": 0.20, "lambda_distill": 0.22}, {}, (42,)),
        ("single_seed (no ensemble)", {}, {}, (42,)),
    ]

    for name, cfg_over, model_over, seeds in variant_specs:
        print(f"\n--- Ablation variant: {name} ---")
        res = run_ablation_variant(name, cfg_overrides=cfg_over, model_overrides=model_over, seeds=seeds)
        ABLATION_RESULTS.append(res)
        print(f"    -> AUC={res['AUC']:.3f}, fold_auc_mean={res['fold_auc_mean']:.3f}, fold_auc_std={res['fold_auc_std']:.3f}")

    ablation_df = pd.DataFrame(ABLATION_RESULTS)
    base_auc = ablation_df.loc[ablation_df["variant"].str.startswith("full_model"), "AUC"].values[0]
    ablation_df["delta_AUC_vs_full"] = ablation_df["AUC"] - base_auc

    print(f"\nAblation study results (folds {ABLATION_FOLD_IDS}, epochs<={ABLATION_MAX_EPOCHS}, "
          f"reduced-budget protocol -- not directly comparable in absolute terms to the full 5-fold/3-seed "
          f"numbers above, only WITHIN this table)")
    display(ablation_df)
else:
    ablation_df = pd.DataFrame()
    print("RUN_ABLATIONS is False; skipping ablation study.")


Running ablation study on folds [1, 2, 3] (of 5 total), max_epochs<=25, patience=6
  Bags: sessions=185, mean_pool_windows=83.4, min=34, max=96
  Bags: sessions=56, mean_pool_windows=76.4, min=31, max=96
  Bags: sessions=65, mean_pool_windows=86.2, min=31, max=96
  Bags: sessions=224, mean_pool_windows=82.1, min=31, max=96
  Bags: sessions=35, mean_pool_windows=88.8, min=43, max=95
  Bags: sessions=47, mean_pool_windows=81.1, min=31, max=96
  Bags: sessions=192, mean_pool_windows=83.3, min=31, max=96
  Bags: sessions=53, mean_pool_windows=79.9, min=37, max=94
  Bags: sessions=61, mean_pool_windows=83.4, min=43, max=94

--- Ablation variant: no_static_fusion ---
    seed 42 epoch 001 | loss=0.7935 | val_auc=0.592 | val_f1=0.765 | sens=0.738 | spec=0.429 | T=0.20 | thr=0.647 | anchor_mass=0.579 | bad=0/6 | elapsed=36.3s
    seed 42 epoch 002 | loss=0.7774 | val_auc=0.609 | val_f1=0.744 | sens=0.690 | spec=0.500 | T=0.20 | thr=0.619 | anchor_mass=0.590 | bad=0/6 | elapsed=70.4s
    seed 4

In [ ]:
# ============================================================
# 8. Save outputs
# ============================================================

stamp = time.strftime("%Y%m%d-%H%M%S")
out_folder = OUT_DIR / f"dro_pas_mil_cbm_experiment_{stamp}"
out_folder.mkdir(parents=True, exist_ok=True)

results_df.to_csv(out_folder / "results_summary.csv", index=False)
session_pred_df.to_csv(out_folder / "session_predictions.csv", index=False)
person_pred_df.to_csv(out_folder / "person_predictions.csv", index=False)
concept_session_df.to_csv(out_folder / "concept_session_values.csv", index=False)
concept_rank.to_csv(out_folder / "concept_rank.csv", index=False)
fold_stability.to_csv(out_folder / "fold_stability.csv", index=False)
if not history_df.empty:
    history_df.to_csv(out_folder / "training_history.csv", index=False)
if not ablation_df.empty:
    ablation_df.to_csv(out_folder / "ablation_results.csv", index=False)
if RUN_BASELINES:
    for bname, bdf in baseline_session_dfs.items():
        bdf.to_csv(out_folder / f"baseline_{bname}_session_predictions.csv", index=False)
    if "stability_df" in globals():
        stability_df.to_csv(out_folder / "fold_stability_all_models.csv", index=False)

with open(out_folder / "significance_tests_vs_baselines.json", "w") as f:
    json.dump(sig_tests, f, indent=2)

with open(out_folder / "config.json", "w") as f:
    json.dump({
        "method": "DRO-PAS-MIL-CBM: Distributionally Robust Multi-Prototype Calibrated Physiology-Anchored MIL-CBM",
        "DATA_DIR": str(DATA_DIR),
        "WINDOW_SIZE": WINDOW_SIZE,
        "OVERLAP": OVERLAP,
        "N_SENSORS": N_SENSORS,
        "MAX_POOL_WINDOWS_PER_SESSION": MAX_POOL_WINDOWS_PER_SESSION,
        "TRAIN_BAG_WINDOWS": TRAIN_BAG_WINDOWS,
        "EVAL_BAG_WINDOWS": EVAL_BAG_WINDOWS,
        "N_ANCHOR_CONCEPTS": N_ANCHOR_CONCEPTS,
        "N_LEARNED_CONCEPTS": N_LEARNED_CONCEPTS,
        "K_CONCEPTS": K_CONCEPTS,
        "HIDDEN": HIDDEN,
        "DEVICE": str(DEVICE),
        "N_PROTOTYPES_PER_CLASS": N_PROTOTYPES_PER_CLASS,
        "DRO_EMA_MOMENTUM": DRO_EMA_MOMENTUM,
        "ENSEMBLE_SEEDS": ENSEMBLE_SEEDS,
        "MIN_VALIDATION_SENSITIVITY": MIN_VALIDATION_SENSITIVITY,
        "BASELINE_MODEL_NAMES": BASELINE_MODEL_NAMES if RUN_BASELINES else [],
        "MAX_EPOCHS_BASELINE": MAX_EPOCHS_BASELINE,
        "PATIENCE_BASELINE": PATIENCE_BASELINE,
        "ABLATION_N_FOLDS": ABLATION_N_FOLDS,
        "ABLATION_MAX_EPOCHS": ABLATION_MAX_EPOCHS,
        "TrainCfg": asdict(cfg),
        "ConceptNames": CONCEPT_NAMES,
        "fold_auc_mean": float(np.mean(fold_auc_vals)),
        "fold_auc_std": float(np.std(fold_auc_vals)),
        "significance_vs_baselines": sig_tests,
    }, f, indent=2)

print("Saved outputs to:", out_folder.resolve())


In [ ]:
# ============================================================
# 9. Paper-method notes
# ============================================================
METHOD_NOTE = """
Each recording is represented as a multi-instance bag and assigned one
session-level label. DRO-PAS-MIL-CBM combines physiological anchors, learned
temporal concepts, prototypes, and static session features. Training uses
EMA-smoothed Group-DRO, temperature scaling, anchor regularization, and a
three-seed ensemble.

CNN1D, ResNet1D, BiLSTM, TCN, and Transformer encoders share the same MIL,
calibration, and threshold-selection protocol. Report fold-level dispersion
and person-clustered bootstrap results with the pooled performance metrics.
"""
print(METHOD_NOTE)


In [ ]:
# ============================================================
# 10. Publication figures
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "legend.frameon": False,
})

figures_dir = out_folder / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)


def _savefig(fig, name):
    fig.savefig(figures_dir / f"{name}.png", bbox_inches="tight")
    fig.savefig(figures_dir / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {name}.png / {name}.pdf")


print(f"\nWriting publication figures to: {figures_dir.resolve()}")

MODEL_COLORS = {
    "DRO-PAS-MIL-CBM": "#1b4965",
    "CNN1D": "#5fa8d3",
    "ResNet1D": "#bee9e8",
    "BiLSTM": "#f4a261",
    "TCN": "#e76f51",
    "Transformer": "#8ab17d",
}

# Assemble a single {model_name: predictions_df} dict for the figure loops below.
all_model_preds = {"DRO-PAS-MIL-CBM": session_pred_df}
if RUN_BASELINES:
    for bname in BASELINE_MODEL_NAMES:
        all_model_preds[bname] = baseline_session_dfs[bname]

# --------------------------------------------------------------------------
# Figure 1: ROC curves, all models overlaid
# --------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.5, 5))
for name, df in all_model_preds.items():
    fpr, tpr, _ = roc_curve(df["label"], df["prob"])
    auc_val = safe_auc(df["label"], df["prob"])
    lw = 2.6 if name == "DRO-PAS-MIL-CBM" else 1.6
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})", color=MODEL_COLORS.get(name), linewidth=lw)
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="Chance")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Session-level ROC curves")
ax.legend(loc="lower right", fontsize=9)
_savefig(fig, "fig01_roc_curves")

# --------------------------------------------------------------------------
# Figure 2: Precision-Recall curves, all models overlaid
# --------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.5, 5))
for name, df in all_model_preds.items():
    prec, rec, _ = precision_recall_curve(df["label"], df["prob"])
    ap_val = safe_ap(df["label"], df["prob"])
    lw = 2.6 if name == "DRO-PAS-MIL-CBM" else 1.6
    ax.plot(rec, prec, label=f"{name} (AP={ap_val:.3f})", color=MODEL_COLORS.get(name), linewidth=lw)
base_rate = df["label"].mean()
ax.axhline(base_rate, linestyle="--", color="gray", linewidth=1, label=f"Prevalence ({base_rate:.2f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Session-level Precision-Recall curves")
ax.legend(loc="lower left", fontsize=9)
_savefig(fig, "fig02_pr_curves")

# --------------------------------------------------------------------------
# Figure 3: Fold-level AUC stability, boxplot across all models
# --------------------------------------------------------------------------
fold_auc_by_model = {}
fold_auc_by_model["DRO-PAS-MIL-CBM"] = fold_stability.set_index("fold")["AUC"].dropna().values
if RUN_BASELINES:
    for bname in BASELINE_MODEL_NAMES:
        bdf = baseline_session_dfs[bname]
        fold_auc_by_model[bname] = bdf.groupby("fold").apply(lambda g: safe_auc(g["label"], g["prob"])).dropna().values

fig, ax = plt.subplots(figsize=(7.5, 5))
labels = list(fold_auc_by_model.keys())
data = [fold_auc_by_model[l] for l in labels]
bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.55, showmeans=True)
for patch, lbl in zip(bp["boxes"], labels):
    patch.set_facecolor(MODEL_COLORS.get(lbl, "#cccccc"))
    patch.set_alpha(0.75)
for lbl_i, lbl in enumerate(labels):
    y = fold_auc_by_model[lbl]
    x = np.random.normal(lbl_i + 1, 0.045, size=len(y))
    ax.scatter(x, y, color="black", s=14, zorder=5, alpha=0.7)
ax.set_ylabel("Session AUC (per test fold)")
ax.set_title("Fold-to-fold AUC stability by model")
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
_savefig(fig, "fig03_fold_auc_stability")

# --------------------------------------------------------------------------
# Figure 4: Calibration / reliability diagram (proposed model vs best baseline)
# --------------------------------------------------------------------------
def per_fold_ece(df, n_bins=5):
    """Per-fold ECE, then averaged across folds.

    The pooled-prediction ECE (computed directly on all folds' sessions
    together) conflates two different things: genuine within-fold
    miscalibration, and between-fold shifts in difficulty/base rate/
    temperature (each fold fits its own temperature on its own validation
    set). Computing ECE separately per fold and then averaging isolates the
    former, which is what "is this model calibrated" should actually mean.
    Uses fewer bins (5) than the plotted diagram since each fold only has
    ~40-65 sessions.
    """
    eces = []
    for _, g in df.groupby("fold"):
        if g["label"].nunique() < 2 or len(g) < n_bins * 2:
            continue
        try:
            t, p = calibration_curve(g["label"], g["prob"], n_bins=n_bins, strategy="quantile")
            eces.append(float(np.mean(np.abs(t - p))))
        except ValueError:
            continue
    if not eces:
        return np.nan, np.nan, 0
    return float(np.mean(eces)), float(np.std(eces)), len(eces)


fig, ax = plt.subplots(figsize=(5.5, 5))
# The curve is pooled; legend ECE values are averaged across folds.
prop_true, prop_pred = calibration_curve(
    session_pred_df["label"], session_pred_df["prob"], n_bins=10, strategy="quantile"
)
ece_prop_mean, ece_prop_std, n_folds_prop = per_fold_ece(session_pred_df)
ax.plot(prop_pred, prop_true, marker="o", color=MODEL_COLORS["DRO-PAS-MIL-CBM"],
        linewidth=2.2, label=f"DRO-PAS-MIL-CBM (ECE\u2248{ece_prop_mean:.3f}\u00b1{ece_prop_std:.3f})")
if RUN_BASELINES:
    best_baseline = max(BASELINE_MODEL_NAMES, key=lambda n: safe_auc(
        baseline_session_dfs[n]["label"], baseline_session_dfs[n]["prob"]))
    bdf = baseline_session_dfs[best_baseline]
    b_true, b_pred = calibration_curve(bdf["label"], bdf["prob"], n_bins=10, strategy="quantile")
    ece_base_mean, ece_base_std, n_folds_base = per_fold_ece(bdf)
    ax.plot(b_pred, b_true, marker="s", color=MODEL_COLORS.get(best_baseline), linewidth=1.6,
            linestyle="--",
            label=f"{best_baseline} (best baseline, ECE\u2248{ece_base_mean:.3f}\u00b1{ece_base_std:.3f})")
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="Perfect calibration")
ax.set_xlabel("Mean predicted probability (per bin)")
ax.set_ylabel("Empirical fraction positive (per bin)")
ax.set_title("Reliability diagram (session-level, after temperature scaling)")
ax.legend(fontsize=8.5, loc="lower right")
ax.text(
    0.02, 0.98,
    "Curve: pooled across folds (visual only).\nECE: mean\u00b1std of per-fold ECE\n(5 quantile bins/fold, avoids conflating\nbetween-fold shifts with true miscalibration)",
    transform=ax.transAxes, fontsize=7, va="top", ha="left", color="dimgray",
    bbox=dict(boxstyle="round", facecolor="white", edgecolor="lightgray", alpha=0.85),
)
_savefig(fig, "fig04_calibration")

# --------------------------------------------------------------------------
# Figure 5: Confusion matrices (session-level and person-level), proposed model
# --------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
for ax, (df, id_col, title) in zip(
    axes,
    [(session_pred_df, "session_id", "Session-level"), (person_pred_df, "person_id", "Person-level")],
):
    cm = confusion_matrix(df["label"], df["pred"], labels=[0, 1])
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Control", "PD"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Control", "PD"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{title} (n={len(df)})")
    vmax = cm.max()
    for i in range(2):
        for j in range(2):
            color = "white" if cm[i, j] > vmax / 2 else "black"
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color=color, fontsize=13, fontweight="bold")
fig.suptitle("DRO-PAS-MIL-CBM confusion matrices (test folds pooled)")
_savefig(fig, "fig05_confusion_matrices")

# --------------------------------------------------------------------------
# Figure 6: Concept relevance ranking (anchors vs learned)
# --------------------------------------------------------------------------
anchor_name_set = set(CONCEPT_NAMES[:N_ANCHOR_CONCEPTS])
cr = concept_rank.sort_values("mean_relevance", ascending=True)
colors = ["#1b4965" if n in anchor_name_set else "#e76f51" for n in cr["concept"]]
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(cr["concept"], cr["mean_relevance"], color=colors)
ax.set_xlabel("Mean relevance weight (sparsemax)")
ax.set_title("Concept relevance ranking")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#1b4965", label="Physiological anchor"),
                    Patch(color="#e76f51", label="Learned concept")], loc="lower right", fontsize=9)
_savefig(fig, "fig06_concept_relevance")

# --------------------------------------------------------------------------
# Figure 7: Class-conditioned distributions for the top anchor concepts
# --------------------------------------------------------------------------
top_anchor_names = [n for n in concept_rank["concept"] if n in anchor_name_set][:4]
fig, axes = plt.subplots(1, len(top_anchor_names), figsize=(4.2 * len(top_anchor_names), 4.2), sharey=False)
if len(top_anchor_names) == 1:
    axes = [axes]
for ax, cname in zip(axes, top_anchor_names):
    vcol = f"{cname}_value"
    ctrl_vals = concept_session_df.loc[concept_session_df["label"] == 0, vcol].values
    pd_vals = concept_session_df.loc[concept_session_df["label"] == 1, vcol].values
    bp = ax.boxplot([ctrl_vals, pd_vals], labels=["Control", "PD"], patch_artist=True, widths=0.5,
                     showmeans=True, showfliers=False)
    for patch, c in zip(bp["boxes"], ["#5fa8d3", "#e76f51"]):
        patch.set_facecolor(c); patch.set_alpha(0.75)
    for i, vals in enumerate([ctrl_vals, pd_vals], start=1):
        x = np.random.normal(i, 0.045, size=len(vals))
        ax.scatter(x, vals, color="black", s=8, alpha=0.35, zorder=5)
    ax.set_title(cname.replace("_", " "), fontsize=10)
fig.suptitle("Top anchor-concept values by class (session-level)")
_savefig(fig, "fig07_concept_value_distributions")

# --------------------------------------------------------------------------
# Figure 8: Ablation study -- accuracy effect vs stability effect
# --------------------------------------------------------------------------
if not ablation_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    order = ablation_df.sort_values("delta_AUC_vs_full")
    colors8 = ["#1b4965" if v.startswith("full_model") else "#e76f51" for v in order["variant"]]
    axes[0].barh(order["variant"], order["delta_AUC_vs_full"], color=colors8)
    axes[0].axvline(0, color="gray", linewidth=1)
    axes[0].set_xlabel("Delta AUC vs. full model")
    axes[0].set_title("Ablation: accuracy effect")

    order2 = ablation_df.sort_values("fold_auc_std")
    colors8b = ["#1b4965" if v.startswith("full_model") else "#e76f51" for v in order2["variant"]]
    axes[1].barh(order2["variant"], order2["fold_auc_std"], color=colors8b)
    axes[1].set_xlabel("Fold AUC std")
    axes[1].set_title("Ablation: stability effect (lower std = more stable)")
    fig.suptitle("Ablation study (reduced-budget protocol, see cell 7b)")
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    _savefig(fig, "fig08_ablation_study")
else:
    print("  skipped fig08_ablation_study (RUN_ABLATIONS was False / ablation_df empty)")

# --------------------------------------------------------------------------
# Figure 9: Average expert-mixture weights (interpretable 4-way decomposition)
# --------------------------------------------------------------------------
mix_cols = ["mix_cbm", "mix_proto", "mix_anchor", "mix_static"]
mix_means = session_pred_df[mix_cols].mean()
mix_labels = ["Sparse CBM", "Prototype", "Anchor-only", "Static fusion"]
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.bar(mix_labels, mix_means.values, color=["#1b4965", "#5fa8d3", "#bee9e8", "#e76f51"])
ax.set_ylabel("Mean mixture weight")
ax.set_title("Average contribution of each expert to the final prediction")
for i, v in enumerate(mix_means.values):
    ax.text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=10)
_savefig(fig, "fig09_mixture_weights")

# --------------------------------------------------------------------------
# Figure 10: Forest plot -- significance vs each SOTA baseline
# --------------------------------------------------------------------------
if RUN_BASELINES and len(sig_tests) > 0:
    names = list(sig_tests.keys())
    diffs = np.array([sig_tests[n]["auc_diff"] for n in names])
    los = np.array([sig_tests[n]["ci95_lo"] for n in names])
    his = np.array([sig_tests[n]["ci95_hi"] for n in names])
    order_idx = np.argsort(diffs)
    names = [names[i] for i in order_idx]
    diffs, los, his = diffs[order_idx], los[order_idx], his[order_idx]
    y_pos = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    colors10 = ["#1b4965" if lo > 0 else ("#e76f51" if hi < 0 else "#999999") for lo, hi in zip(los, his)]
    for i, (d, lo, hi, c) in enumerate(zip(diffs, los, his, colors10)):
        ax.errorbar([d], [i], xerr=[[d - lo], [hi - d]], fmt="o", color="black",
                    ecolor=c, elinewidth=3, capsize=4, markersize=6)
    ax.axvline(0, color="gray", linestyle="--", linewidth=1)
    ax.set_yticks(y_pos); ax.set_yticklabels(names)
    ax.set_xlabel("AUC diff (DRO-PAS-MIL-CBM minus baseline), 95% CI")
    ax.set_title("Significance vs. each SOTA baseline (person-clustered bootstrap)")
    _savefig(fig, "fig10_significance_forest_plot")
else:
    print("  skipped fig10_significance_forest_plot (RUN_BASELINES was False)")

# --------------------------------------------------------------------------
# Figure 11: Training curves (proposed model), mean +/- std across folds/seeds
# --------------------------------------------------------------------------
if not history_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    g = history_df.groupby("epoch")
    ep = g.size().index.values
    for ax, col, ylabel, color in [
        (axes[0], "train_loss", "Training loss", "#e76f51"),
        (axes[1], "val_auc", "Validation AUC", "#1b4965"),
    ]:
        mean_v = g[col].mean()
        std_v = g[col].std()
        ax.plot(ep, mean_v.values, color=color, linewidth=2)
        ax.fill_between(ep, (mean_v - std_v).values, (mean_v + std_v).values, color=color, alpha=0.2)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
    fig.suptitle("Training curves: mean +/- std across all folds x seeds")
    fig.tight_layout()
    _savefig(fig, "fig11_training_curves")
else:
    print("  skipped fig11_training_curves (history_df empty)")

print(f"\nAll figures written to: {figures_dir.resolve()}")
